In [ ]:
# Cell 1 — Imports + shared constants
# Import the main packages, set simple display options, and define the shared labels and groups used later.

from pathlib import Path  # Use clear and safe file-system paths.
from datetime import datetime  # Create timestamp-based output-folder names.
import json  # Save and load small settings files.
import ast  # Turn saved text dictionaries back into real Python objects.
import itertools  # Build simple pairwise group comparisons later.

import numpy as np  # Work with arrays and numeric values.
import pandas as pd  # Read CSV files and organize tables.
import zarr  # Open dataset.zarr without loading everything into memory.
import matplotlib.pyplot as plt  # Display images and create plots.

import ipywidgets as widgets  # Build simple interactive controls inside the notebook.
from IPython.display import display, HTML, clear_output  # Show tables, widgets, and refreshed outputs.

from sklearn.metrics import confusion_matrix  # Build confusion-matrix counts from true and predicted labels.

pd.set_option("display.max_columns", 200)  # Show many columns when wide tables are displayed.
pd.set_option("display.max_rows", 200)  # Show enough rows to inspect selected tables.
pd.set_option("display.width", 160)  # Reduce line wrapping when tables are printed.
plt.rcParams["figure.dpi"] = 140  # Use a readable default figure resolution in the notebook.

CLASS_ORDER = ["gray_d", "gray_l", "red", "green", "blue", "yellow"]  # Keep the same class order used in your dataset and CNN workflow.
LABEL_TO_INT = {label: idx for idx, label in enumerate(CLASS_ORDER)}  # Convert each text label into its integer class ID.
INT_TO_LABEL = {idx: label for label, idx in LABEL_TO_INT.items()}  # Convert each integer class ID back into its text label.

GROUP_ORDER = ["Classic4", "Classic4_plus_s", "Neitz8"]  # Keep the three CNN groups in one fixed display order.
EVAL_OPTIONS = ["test_best", "test_final"]  # Allow the user to choose which saved CNN test results to use later.

KEY_TO_LABEL = {"a": "red", "s": "green", "d": "blue", "f": "yellow", "k": "gray_d", "l": "gray_l"}  # Map keyboard keys to labels for optional human review.
LABEL_TO_KEY = {label: key for key, label in KEY_TO_LABEL.items()}  # Build the reverse mapping from label back to keyboard key.

print("Cell 1 loaded successfully.")  # Confirm that the setup cell ran.
print("Class order:", CLASS_ORDER)  # Show the label order that later tables and confusion matrices will use.
print("Group order:", GROUP_ORDER)  # Show the group order that later comparisons will use.
print("Evaluation choices:", EVAL_OPTIONS)  # Show the two CNN test-result options that will be offered later.

In [ ]:
# Cell 2 — Paths + file checks
# Define the shared remote-server paths, point to the CNN result files, and confirm that the needed files exist.

PROJECT_DIR = Path(".").resolve()  # Use the notebook's current folder as the main project folder on the remote server.
DATASET_RUN_DIR = PROJECT_DIR / "export_retina_20260313_230715"  # Point to the shared dataset run folder used by the CNN notebook.
CNN_EVAL_DIR = DATASET_RUN_DIR / "cnn_eval_outputs_20260315_094615"  # Point to the saved CNN evaluation-output folder inside the dataset run folder.
EXISTING_REVIEW_DIR = PROJECT_DIR / "multi_run_review_20260318_190013"  # Point to the earlier review notebook output folder that already contains human labels.

ZARR_PATH = DATASET_RUN_DIR / "dataset.zarr"  # Point to the shared stimulus-image dataset.
META_PATH = DATASET_RUN_DIR / "metadata.csv"  # Point to the metadata table for the shared stimulus dataset.
CNN_TRAIN_SUMMARY_PATH = CNN_EVAL_DIR / "tables" / "cnn_train_summary.csv"  # Point to the main CNN run-summary table.
CNN_EVAL_SUMMARY_PATH = CNN_EVAL_DIR / "tables" / "cnn_eval_summary.csv"  # Point to the optional CNN evaluation-summary table.
CNN_SPLIT_PATH = CNN_EVAL_DIR / "splits" / "split_indices.csv"  # Point to the CNN experiment's own train/val/test split table.
CNN_HISTORIES_PATH = CNN_EVAL_DIR / "histories" / "all_histories_epochs.csv"  # Point to the per-epoch CNN history table.
CNN_ARTIFACT_MANIFEST_PATH = CNN_EVAL_DIR / "artifacts" / "artifact_files.csv"  # Point to the artifact manifest that records saved CNN files.
CNN_ARTIFACTS_ROOT = DATASET_RUN_DIR / "cnn_artifacts" / "multi_run_experiments"  # Point to the real remote-server folder that contains the saved .npz prediction files.

EXISTING_CACHE_PATH = EXISTING_REVIEW_DIR / "cache" / "review_cache_master.csv"  # Point to the notebook-level human-label cache from the earlier review notebook.
EXISTING_SCOPE_DIR = EXISTING_REVIEW_DIR / "review_scopes"  # Point to the folder that contains per-scope human-label CSV files.

NOTEBOOK_OUTPUT_PARENT = PROJECT_DIR  # Save new notebook outputs beside the other notebooks and result folders.
DEFAULT_OUTPUT_DIR_NAME = "cnn_multi_run_review_" + datetime.now().strftime("%Y%m%d_%H%M%S")  # Create a simple default folder name for this new notebook's outputs.

REQUIRED_PATHS = {  # Collect the files and folders that must exist for this notebook to work correctly.
    "PROJECT_DIR": PROJECT_DIR,  # Keep the project folder itself in the required-path checks.
    "DATASET_RUN_DIR": DATASET_RUN_DIR,  # Require the shared dataset run folder.
    "ZARR_PATH": ZARR_PATH,  # Require the shared image dataset.
    "META_PATH": META_PATH,  # Require the shared metadata table.
    "CNN_EVAL_DIR": CNN_EVAL_DIR,  # Require the saved CNN evaluation-output folder.
    "CNN_TRAIN_SUMMARY_PATH": CNN_TRAIN_SUMMARY_PATH,  # Require the main CNN summary CSV.
    "CNN_SPLIT_PATH": CNN_SPLIT_PATH,  # Require the saved CNN split table.
    "CNN_HISTORIES_PATH": CNN_HISTORIES_PATH,  # Require the saved CNN history table.
    "CNN_ARTIFACT_MANIFEST_PATH": CNN_ARTIFACT_MANIFEST_PATH,  # Require the saved artifact manifest.
    "CNN_ARTIFACTS_ROOT": CNN_ARTIFACTS_ROOT,  # Require the real remote-server CNN artifact root where the .npz files live.
}  # End of required-path dictionary.

OPTIONAL_PATHS = {  # Collect the files and folders that are optional but useful for reusing human-reviewed labels.
    "EXISTING_REVIEW_DIR": EXISTING_REVIEW_DIR,  # Earlier review-notebook output folder.
    "EXISTING_CACHE_PATH": EXISTING_CACHE_PATH,  # Notebook-level human-label cache file.
    "EXISTING_SCOPE_DIR": EXISTING_SCOPE_DIR,  # Folder that stores per-scope human-label CSV files.
    "CNN_EVAL_SUMMARY_PATH": CNN_EVAL_SUMMARY_PATH,  # Optional CNN evaluation-summary CSV.
}  # End of optional-path dictionary.

for path_name, path_value in REQUIRED_PATHS.items():  # Check each required file or folder one at a time.
    if not path_value.exists():  # Stop immediately if any required path is missing.
        raise FileNotFoundError(f"Required path not found: {path_name} -> {path_value}")  # Give a clear error message that shows the missing path.

path_rows = []  # Collect one summary row for each checked path so the user can inspect them easily.

for path_name, path_value in REQUIRED_PATHS.items():  # Add all required paths to the summary table.
    path_rows.append({"path_name": path_name, "path_value": str(path_value), "required": True, "exists": path_value.exists()})  # Save one row that describes this required path.

for path_name, path_value in OPTIONAL_PATHS.items():  # Add all optional paths to the same summary table.
    path_rows.append({"path_name": path_name, "path_value": str(path_value), "required": False, "exists": path_value.exists()})  # Save one row that describes this optional path.

PATHS_DF = pd.DataFrame(path_rows)  # Convert the collected path rows into one easy-to-read table.
display(PATHS_DF)  # Show the full path table in the notebook output.

print("All required paths were found.")  # Confirm that the main CNN and dataset files are available.
print("Default new output folder name:", DEFAULT_OUTPUT_DIR_NAME)  # Show the default folder name for outputs from this new notebook.
print("Existing human-label cache found:", EXISTING_CACHE_PATH.exists())  # Show whether the notebook-level human-label cache is available to reuse.
print("Existing review-scope folder found:", EXISTING_SCOPE_DIR.exists())  # Show whether the earlier per-scope human-label files are available to reuse.

In [ ]:
# Cell 3 — User choices  # 03.26.2026 Add the new top-of-notebook controls for gray handling, human review, saved-result reloading, and output location.
# Let the user choose which CNN test results to use, which groups to compare, how to handle gray classes, whether to use human review, whether to reuse prior human labels, and whether to reload saved results only.  # 03.26.2026 Explain the expanded purpose of this choice cell.

eval_choice_widget = widgets.Dropdown(  # 03.26.2026 Keep the evaluation-choice dropdown for test_best versus test_final selection.
    options=EVAL_OPTIONS,  # 03.26.2026 Offer the two saved CNN evaluation choices defined in Cell 1.
    value="test_best",  # 03.26.2026 Start with test_best as the default evaluation choice.
    description="Results:",  # 03.26.2026 Label the evaluation-choice dropdown clearly.
    style={"description_width": "initial"},  # 03.26.2026 Keep the full widget description visible.
)  # 03.26.2026 Finish the evaluation-choice widget.

group_choice_widget = widgets.SelectMultiple(  # 03.26.2026 Keep the multi-select widget for choosing one or more CNN groups.
    options=GROUP_ORDER,  # 03.26.2026 Offer the fixed CNN group order defined in Cell 1.
    value=tuple(GROUP_ORDER),  # 03.26.2026 Start with all CNN groups selected.
    description="Groups:",  # 03.26.2026 Label the group selector clearly.
    rows=len(GROUP_ORDER),  # 03.26.2026 Show one row for each available CNN group.
    style={"description_width": "initial"},  # 03.26.2026 Keep the full widget description visible.
)  # 03.26.2026 Finish the group-choice widget.

combine_gray_classes_widget = widgets.Checkbox(  # 03.26.2026 Add the requested control for merging gray_d and gray_l into one gray evaluation class.
    value=True,  # 03.26.2026 Start with gray merging enabled because the most recent workflow used that option.
    description="Combine gray_d and gray_l into gray",  # 03.26.2026 Explain the gray-handling control directly.
    indent=False,  # 03.26.2026 Keep the checkbox aligned cleanly with the other controls.
)  # 03.26.2026 Finish the gray-handling checkbox.

use_human_review_widget = widgets.Checkbox(  # 03.26.2026 Add the requested control for choosing human review versus automated labels only.
    value=False,  # 03.26.2026 Start in automated-label mode so the notebook can run without manual review by default.
    description="Use human review for shared misclassifications",  # 03.26.2026 Explain the label-source control directly.
    indent=False,  # 03.26.2026 Keep the checkbox aligned cleanly with the other controls.
)  # 03.26.2026 Finish the human-review checkbox.

use_previous_human_labels_widget = widgets.Checkbox(  # 03.26.2026 Add the requested control for reusing previously generated human labels when human review is enabled.
    value=True,  # 03.26.2026 Start with prior human-label reuse enabled because that matches the earlier notebook behavior.
    description="Reuse previously generated human-review labels",  # 03.26.2026 Explain the prior-label reuse control directly.
    indent=False,  # 03.26.2026 Keep the checkbox aligned cleanly with the other controls.
)  # 03.26.2026 Finish the prior-label reuse checkbox.

load_saved_results_only_widget = widgets.Checkbox(  # 03.26.2026 Add the requested control for skipping analysis and reloading previously saved notebook outputs only.
    value=False,  # 03.26.2026 Start in full-analysis mode so the notebook behaves normally unless reload-only mode is requested.
    description="Skip analysis and reload previous saved data only",  # 03.26.2026 Explain the saved-result reload control directly.
    indent=False,  # 03.26.2026 Keep the checkbox aligned cleanly with the other controls.
)  # 03.26.2026 Finish the reload-only checkbox.

output_dir_widget = widgets.Text(  # 03.26.2026 Keep the editable output-folder text box for the new notebook file.
    value=DEFAULT_OUTPUT_DIR_NAME,  # 03.26.2026 Start with the default timestamped output-folder name from Cell 2.
    description="Output folder:",  # 03.26.2026 Label the output-folder text box clearly.
    style={"description_width": "initial"},  # 03.26.2026 Keep the full widget description visible.
    layout=widgets.Layout(width="700px"),  # 03.26.2026 Make the text box wide enough to show the full folder name.
)  # 03.26.2026 Finish the output-folder text widget.

apply_choices_button = widgets.Button(  # 03.26.2026 Keep the one-click button that stores the selected notebook settings.
    description="Save choices",  # 03.26.2026 Use the same simple action label as before.
    button_style="primary",  # 03.26.2026 Highlight the save button so it is easy to find.
)  # 03.26.2026 Finish the save-choices button.

choices_output = widgets.Output()  # 03.26.2026 Keep the output area that shows warnings and the saved settings summary.
USER_CONFIG = {}  # 03.26.2026 Reset the shared configuration dictionary before the user saves new settings.

def refresh_human_review_widgets(*_args):  # 03.26.2026 Keep the prior-label reuse control synchronized with the human-review choice.
    use_previous_human_labels_widget.disabled = not bool(use_human_review_widget.value)  # 03.26.2026 Disable prior-label reuse when human review is turned off.
    if not bool(use_human_review_widget.value):  # 03.26.2026 Reset the prior-label reuse checkbox when human review is disabled.
        use_previous_human_labels_widget.value = False  # 03.26.2026 Make the disabled prior-label reuse setting explicit.

def save_user_choices(_button):  # 03.26.2026 Save the requested notebook settings into one shared configuration dictionary.
    selected_groups = list(group_choice_widget.value)  # 03.26.2026 Convert the selected CNN groups into a regular Python list.
    if len(selected_groups) == 0:  # 03.26.2026 Require at least one selected group before saving the configuration.
        with choices_output:  # 03.26.2026 Send the warning into the saved-settings output area.
            clear_output()  # 03.26.2026 Clear any earlier status message before showing the new warning.
            print("Please select at least one group.")  # 03.26.2026 Tell the user how to fix the missing-group problem.
        return  # 03.26.2026 Stop here until at least one group is selected.

    output_dir_name = output_dir_widget.value.strip()  # 03.26.2026 Read and clean the chosen output-folder name.
    if output_dir_name == "":  # 03.26.2026 Require a non-empty output-folder name before saving the configuration.
        with choices_output:  # 03.26.2026 Send the warning into the saved-settings output area.
            clear_output()  # 03.26.2026 Clear any earlier status message before showing the new warning.
            print("Please enter an output folder name.")  # 03.26.2026 Tell the user how to fix the blank output-folder problem.
        return  # 03.26.2026 Stop here until a valid output-folder name is entered.

    combine_gray_classes = bool(combine_gray_classes_widget.value)  # 03.26.2026 Read the requested gray-handling mode from the new checkbox.
    use_human_review = bool(use_human_review_widget.value)  # 03.26.2026 Read the requested label-source mode from the new checkbox.
    use_previous_human_labels = bool(use_previous_human_labels_widget.value) if use_human_review else False  # 03.26.2026 Reuse prior human labels only when human review mode is enabled.
    load_saved_results_only = bool(load_saved_results_only_widget.value)  # 03.26.2026 Read the requested reload-only mode from the new checkbox.
    eval_class_order = ["gray", "red", "green", "blue", "yellow"] if combine_gray_classes else list(CLASS_ORDER)  # 03.26.2026 Build the active evaluation class order from the gray-handling choice.
    eval_mode_suffix = "gray_merged" if combine_gray_classes else "separate_gray"  # 03.26.2026 Build one short text label for the active gray-handling mode.
    label_source_suffix = "human_review" if use_human_review else "auto_labels"  # 03.26.2026 Build one short text label for the active label-source mode.
    notebook_summary_file_name = f"simple_summary_human_review_long_{eval_mode_suffix}.csv" if use_human_review else ("simple_summary_original_long_gray_merged.csv" if combine_gray_classes else "simple_summary_original_long.csv")  # 03.26.2026 Define the notebook-level long-summary filename for the active evaluation mode.
    eval_output_folder_name = "gray_merged_outputs" if ((not use_human_review) and combine_gray_classes) else ("original_outputs" if ((not use_human_review) and (not combine_gray_classes)) else f"{label_source_suffix}_{eval_mode_suffix}_outputs")  # 03.26.2026 Define the active figure-output folder name while preserving the existing gray-merged folder name.

    USER_CONFIG["eval_choice"] = str(eval_choice_widget.value)  # 03.26.2026 Save the chosen CNN evaluation type.
    USER_CONFIG["groups"] = selected_groups  # 03.26.2026 Save the chosen CNN groups.
    USER_CONFIG["combine_gray_classes"] = combine_gray_classes  # 03.26.2026 Save the gray-handling mode.
    USER_CONFIG["use_human_review"] = use_human_review  # 03.26.2026 Save the label-source mode.
    USER_CONFIG["use_previous_human_labels"] = use_previous_human_labels  # 03.26.2026 Save whether previously generated human labels should be reused.
    USER_CONFIG["load_saved_results_only"] = load_saved_results_only  # 03.26.2026 Save whether the notebook should skip new analysis and reload saved files only.
    USER_CONFIG["eval_class_order"] = eval_class_order  # 03.26.2026 Save the active evaluation class order for downstream cells.
    USER_CONFIG["eval_mode_suffix"] = eval_mode_suffix  # 03.26.2026 Save the short gray-handling mode label for downstream filenames.
    USER_CONFIG["label_source_suffix"] = label_source_suffix  # 03.26.2026 Save the short label-source mode label for downstream filenames.
    USER_CONFIG["notebook_summary_file_name"] = notebook_summary_file_name  # 03.26.2026 Save the active notebook-level long-summary filename.
    USER_CONFIG["eval_output_folder_name"] = eval_output_folder_name  # 03.26.2026 Save the active figure-output folder name.
    USER_CONFIG["output_dir_name"] = output_dir_name  # 03.26.2026 Save the chosen output-folder name.
    USER_CONFIG["output_dir"] = NOTEBOOK_OUTPUT_PARENT / output_dir_name  # 03.26.2026 Save the full output-folder path as a Path object.

    global SELECTED_GROUPS, EVAL_CHOICE, NOTEBOOK_OUTPUT_DIR  # 03.26.2026 Expose the core saved settings to later cells, including reload-only mode.
    SELECTED_GROUPS = list(selected_groups)  # 03.26.2026 Keep the selected CNN groups available as a direct global variable.
    EVAL_CHOICE = str(USER_CONFIG["eval_choice"])  # 03.26.2026 Keep the chosen CNN evaluation type available as a direct global variable.
    NOTEBOOK_OUTPUT_DIR = Path(USER_CONFIG["output_dir"])  # 03.26.2026 Keep the notebook output directory available as a direct global variable.

    with choices_output:  # 03.26.2026 Print the saved configuration in the output area below the widgets.
        clear_output()  # 03.26.2026 Clear any earlier status message before showing the saved settings.
        print("Choices saved successfully.")  # 03.26.2026 Confirm that the configuration was stored.
        print(json.dumps({key: (str(value) if isinstance(value, Path) else value) for key, value in USER_CONFIG.items()}, indent=2))  # 03.26.2026 Show the full saved configuration in a readable JSON-like format.

use_human_review_widget.observe(refresh_human_review_widgets, names="value")  # 03.26.2026 Refresh the prior-label reuse control whenever the human-review choice changes.
refresh_human_review_widgets()  # 03.26.2026 Apply the initial enabled or disabled state for the prior-label reuse control.
apply_choices_button.on_click(save_user_choices)  # 03.26.2026 Connect the save button to the configuration-saving callback.

display(HTML("<b>Choose the CNN comparison settings below, then click <i>Save choices</i>.</b>"))  # 03.26.2026 Show a short instruction line above the expanded controls.
display(eval_choice_widget)  # 03.26.2026 Show the evaluation-choice dropdown.
display(group_choice_widget)  # 03.26.2026 Show the group-selection widget.
display(combine_gray_classes_widget)  # 03.26.2026 Show the new gray-handling checkbox.
display(use_human_review_widget)  # 03.26.2026 Show the new human-review checkbox.
display(use_previous_human_labels_widget)  # 03.26.2026 Show the new prior-label reuse checkbox.
display(load_saved_results_only_widget)  # 03.26.2026 Show the new reload-only checkbox.
display(output_dir_widget)  # 03.26.2026 Show the output-folder text widget.
display(apply_choices_button)  # 03.26.2026 Show the save-choices button.
display(choices_output)  # 03.26.2026 Show the saved-settings output area.


In [ ]:
# Cell 4 — Load metadata  # 03.26.2026 Support the new reload-only mode while keeping the metadata-loading path unchanged for fresh analysis runs.
# Read the shared stimulus metadata file, check the key columns, and prepare a clean metadata table for later merges when fresh analysis is enabled.  # 03.26.2026 Explain the updated purpose of this cell.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

SELECTED_GROUPS = list(USER_CONFIG["groups"])  # 03.26.2026 Keep the selected CNN groups available as a direct global variable in every mode.
EVAL_CHOICE = str(USER_CONFIG["eval_choice"])  # 03.26.2026 Keep the chosen CNN evaluation type available as a direct global variable in every mode.
NOTEBOOK_OUTPUT_DIR = Path(USER_CONFIG["output_dir"])  # 03.26.2026 Keep the chosen output directory available as a direct global variable in every mode.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip metadata loading when the notebook is set to reload saved outputs only.
    META_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder metadata table in reload-only mode because later figure-rebuild cells do not need metadata rows.
    print("Load-saved-results-only is enabled, so Cell 4 skipped metadata loading.")  # 03.26.2026 Confirm that metadata loading was skipped intentionally.
else:  # 03.26.2026 Run the normal metadata-loading path when fresh analysis is enabled.
    META_DF = pd.read_csv(META_PATH)  # 03.26.2026 Read the metadata CSV into a pandas DataFrame.
    REQUIRED_META_COLUMNS = ["stim_ID", "true_label"]  # 03.26.2026 Keep the required metadata columns needed later in the notebook.
    for column_name in REQUIRED_META_COLUMNS:  # 03.26.2026 Check each required metadata column one at a time.
        if column_name not in META_DF.columns:  # 03.26.2026 Stop if a required metadata column is missing.
            raise KeyError(f"Required metadata column not found: {column_name}")  # 03.26.2026 Name the missing metadata column clearly.
    META_DF = META_DF.copy()  # 03.26.2026 Make an explicit copy so later edits stay local to this notebook.
    META_DF["stim_ID"] = pd.to_numeric(META_DF["stim_ID"], errors="raise").astype(int)  # 03.26.2026 Force stim_ID to integer type for safe joins.
    META_DF["true_label"] = META_DF["true_label"].astype(str).str.strip()  # 03.26.2026 Normalize the text true labels.
    META_DF["true_label_int"] = META_DF["true_label"].map(LABEL_TO_INT)  # 03.26.2026 Convert text true labels into integer class IDs.
    if META_DF["true_label_int"].isna().any():  # 03.26.2026 Stop if any metadata label is not part of CLASS_ORDER.
        bad_labels = sorted(META_DF.loc[META_DF["true_label_int"].isna(), "true_label"].unique().tolist())  # 03.26.2026 Collect the unexpected metadata labels for the error message.
        raise ValueError(f"Metadata contains labels that are not in CLASS_ORDER: {bad_labels}")  # 03.26.2026 Explain which metadata labels caused the failure.
    META_DF["stimulus_index"] = META_DF["stim_ID"]  # 03.26.2026 Keep a CNN-style stimulus_index column for later merges.
    META_DF["stimulus_key"] = META_DF["stim_ID"].astype(str)  # 03.26.2026 Keep a string stimulus key for safe joins to saved review labels.
    META_COLUMNS_TO_SHOW = [column_name for column_name in ["stim_ID", "stimulus_index", "true_label", "true_label_int", "bg_hue", "bg_int", "obj_int", "delta_int", "bg_sat", "obj_sat", "delta_sat"] if column_name in META_DF.columns]  # 03.26.2026 Keep the same small metadata preview set when those columns exist.
    print("Metadata loaded successfully.")  # 03.26.2026 Confirm that the metadata table was read and cleaned.
    print("Number of stimuli:", len(META_DF))  # 03.26.2026 Show how many total stimuli are in the metadata table.
    print("Number of metadata columns:", len(META_DF.columns))  # 03.26.2026 Show how many columns are available in the metadata table.
    print("Unique labels:", sorted(META_DF["true_label"].unique().tolist()))  # 03.26.2026 Show the unique class labels found in the metadata.
    display(META_DF[META_COLUMNS_TO_SHOW].head(20))  # 03.26.2026 Preview the first few cleaned metadata rows.


In [ ]:
# Cell 5 — Load CNN run tables  # 03.26.2026 Support the new reload-only mode while keeping the CNN-summary loading path unchanged for fresh analysis runs.
# Read the CNN summary tables, keep only the chosen groups, and define the exact summary columns that match the selected test result type when fresh analysis is enabled.  # 03.26.2026 Explain the updated purpose of this cell.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

SELECTED_GROUPS = list(USER_CONFIG["groups"])  # 03.26.2026 Keep the selected CNN groups available as a direct global variable in every mode.
EVAL_CHOICE = str(USER_CONFIG["eval_choice"])  # 03.26.2026 Keep the chosen CNN evaluation type available as a direct global variable in every mode.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip fresh CNN table loading when the notebook is set to reload saved outputs only.
    CNN_TRAIN_SUMMARY_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder CNN train-summary table in reload-only mode.
    CNN_SPLIT_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder split table in reload-only mode.
    CNN_HISTORIES_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder history table in reload-only mode.
    CNN_RUNS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder selected-run table in reload-only mode.
    TEST_SPLIT_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder test-split table in reload-only mode.
    TEST_STIMULUS_IDS = []  # 03.26.2026 Use an empty placeholder test-stimulus list in reload-only mode.
    print("Load-saved-results-only is enabled, so Cell 5 skipped CNN table loading.")  # 03.26.2026 Confirm that CNN table loading was skipped intentionally.
else:  # 03.26.2026 Run the normal CNN-summary loading path when fresh analysis is enabled.
    CNN_TRAIN_SUMMARY_DF = pd.read_csv(CNN_TRAIN_SUMMARY_PATH)  # 03.26.2026 Read the main CNN run-summary CSV.
    CNN_SPLIT_DF = pd.read_csv(CNN_SPLIT_PATH)  # 03.26.2026 Read the CNN train-validation-test split table.
    CNN_HISTORIES_DF = pd.read_csv(CNN_HISTORIES_PATH)  # 03.26.2026 Read the per-epoch CNN training-history table.
    REQUIRED_TRAIN_SUMMARY_COLUMNS = ["group", "split_seed", "trial", "best_epoch", "best_val_acc", "test_acc_best", "test_acc_final", "test_artifacts_dir_best", "test_artifacts_dir_final"]  # 03.26.2026 Keep the required run-summary columns used later in the notebook.
    REQUIRED_SPLIT_COLUMNS = ["split_seed", "subset", "position_in_subset", "stimulus_index"]  # 03.26.2026 Keep the required split-table columns used later in the notebook.
    REQUIRED_HISTORY_COLUMNS = ["group", "split_seed", "trial_index", "epoch", "train_loss", "train_acc", "val_loss", "val_acc", "epoch_sec"]  # 03.26.2026 Keep the required history-table columns used later in the notebook.
    for column_name in REQUIRED_TRAIN_SUMMARY_COLUMNS:  # 03.26.2026 Check each required run-summary column one at a time.
        if column_name not in CNN_TRAIN_SUMMARY_DF.columns:  # 03.26.2026 Stop if a required run-summary column is missing.
            raise KeyError(f"Required CNN train-summary column not found: {column_name}")  # 03.26.2026 Name the missing run-summary column clearly.
    for column_name in REQUIRED_SPLIT_COLUMNS:  # 03.26.2026 Check each required split-table column one at a time.
        if column_name not in CNN_SPLIT_DF.columns:  # 03.26.2026 Stop if a required split-table column is missing.
            raise KeyError(f"Required CNN split column not found: {column_name}")  # 03.26.2026 Name the missing split-table column clearly.
    for column_name in REQUIRED_HISTORY_COLUMNS:  # 03.26.2026 Check each required history-table column one at a time.
        if column_name not in CNN_HISTORIES_DF.columns:  # 03.26.2026 Stop if a required history-table column is missing.
            raise KeyError(f"Required CNN history column not found: {column_name}")  # 03.26.2026 Name the missing history-table column clearly.
    CNN_TRAIN_SUMMARY_DF = CNN_TRAIN_SUMMARY_DF.copy()  # 03.26.2026 Make an explicit copy so later edits stay local to this notebook.
    CNN_SPLIT_DF = CNN_SPLIT_DF.copy()  # 03.26.2026 Make an explicit copy of the split table for safe later edits.
    CNN_HISTORIES_DF = CNN_HISTORIES_DF.copy()  # 03.26.2026 Make an explicit copy of the history table for safe later edits.
    CNN_TRAIN_SUMMARY_DF["group"] = CNN_TRAIN_SUMMARY_DF["group"].astype(str).str.strip()  # 03.26.2026 Clean the group names in the train-summary table.
    CNN_SPLIT_DF["subset"] = CNN_SPLIT_DF["subset"].astype(str).str.strip()  # 03.26.2026 Clean the subset names in the split table.
    CNN_SPLIT_DF["stimulus_index"] = pd.to_numeric(CNN_SPLIT_DF["stimulus_index"], errors="raise").astype(int)  # 03.26.2026 Force stimulus_index to integer type for safe joins.
    CNN_HISTORIES_DF["group"] = CNN_HISTORIES_DF["group"].astype(str).str.strip()  # 03.26.2026 Clean the group names in the history table.
    CNN_RUNS_DF = CNN_TRAIN_SUMMARY_DF[CNN_TRAIN_SUMMARY_DF["group"].isin(SELECTED_GROUPS)].copy()  # 03.26.2026 Keep only the selected groups in the run-summary table.
    CNN_HISTORIES_SELECTED_DF = CNN_HISTORIES_DF[CNN_HISTORIES_DF["group"].isin(SELECTED_GROUPS)].copy()  # 03.26.2026 Keep only the selected groups in the history table.
    if len(CNN_RUNS_DF) == 0:  # 03.26.2026 Stop if the selected CNN groups produced no matching run-summary rows.
        raise ValueError(f"No CNN runs were found for the selected groups: {SELECTED_GROUPS}")  # 03.26.2026 Explain which selected groups had no matching runs.
    if EVAL_CHOICE == "test_best":  # 03.26.2026 Use the correct saved-summary columns for the best-validation checkpoint option.
        CHOSEN_TEST_ACC_COL = "test_acc_best"  # 03.26.2026 Keep the accuracy column name that matches test_best.
        CHOSEN_TEST_LOSS_COL = "test_loss_best"  # 03.26.2026 Keep the loss column name that matches test_best.
        CHOSEN_TEST_ARTIFACT_DIR_COL = "test_artifacts_dir_best"  # 03.26.2026 Keep the artifact-directory column that matches test_best.
    elif EVAL_CHOICE == "test_final":  # 03.26.2026 Use the correct saved-summary columns for the final-epoch checkpoint option.
        CHOSEN_TEST_ACC_COL = "test_acc_final"  # 03.26.2026 Keep the accuracy column name that matches test_final.
        CHOSEN_TEST_LOSS_COL = "test_loss_final"  # 03.26.2026 Keep the loss column name that matches test_final.
        CHOSEN_TEST_ARTIFACT_DIR_COL = "test_artifacts_dir_final"  # 03.26.2026 Keep the artifact-directory column that matches test_final.
    else:  # 03.26.2026 Stop if the saved evaluation choice is not recognized.
        raise ValueError(f"Unsupported evaluation choice: {EVAL_CHOICE}")  # 03.26.2026 Show the unexpected evaluation choice clearly.
    CNN_RUNS_DF["chosen_test_acc"] = CNN_RUNS_DF[CHOSEN_TEST_ACC_COL]  # 03.26.2026 Copy the selected test-accuracy values into one shared column.
    CNN_RUNS_DF["chosen_test_loss"] = CNN_RUNS_DF[CHOSEN_TEST_LOSS_COL]  # 03.26.2026 Copy the selected test-loss values into one shared column.
    CNN_RUNS_DF["chosen_test_artifacts_dir"] = CNN_RUNS_DF[CHOSEN_TEST_ARTIFACT_DIR_COL].astype(str)  # 03.26.2026 Copy the selected test artifact folder into one shared column.
    TEST_SPLIT_DF = CNN_SPLIT_DF[CNN_SPLIT_DF["subset"].str.lower() == "test"].copy()  # 03.26.2026 Keep only the CNN experiment's final test split.
    TEST_STIMULUS_IDS = sorted(TEST_SPLIT_DF["stimulus_index"].unique().tolist())  # 03.26.2026 Collect the unique CNN test stimulus IDs into one sorted list.
    RUN_PREVIEW_COLUMNS = [column_name for column_name in ["group", "split_seed", "trial", "init_seed", "best_epoch", "best_val_acc", CHOSEN_TEST_ACC_COL, CHOSEN_TEST_ARTIFACT_DIR_COL, "chosen_test_acc", "chosen_test_artifacts_dir"] if column_name in CNN_RUNS_DF.columns]  # 03.26.2026 Keep the same compact run-summary preview columns.
    print("CNN run tables loaded successfully.")  # 03.26.2026 Confirm that the CNN summary tables were read and cleaned.
    print("Selected evaluation type:", EVAL_CHOICE)  # 03.26.2026 Show whether the notebook will use test_best or test_final outputs.
    print("Selected groups:", SELECTED_GROUPS)  # 03.26.2026 Show which CNN groups are currently included.
    print("Number of selected CNN runs:", len(CNN_RUNS_DF))  # 03.26.2026 Show how many selected run-summary rows are included.
    print("Number of test stimuli in the CNN split:", len(TEST_STIMULUS_IDS))  # 03.26.2026 Show how many unique stimuli are in the final CNN test set.
    display(CNN_RUNS_DF[RUN_PREVIEW_COLUMNS].sort_values(["group", "split_seed", "trial"]).reset_index(drop=True))  # 03.26.2026 Show the selected run summaries in a stable readable order.


In [ ]:
# Cell 6 — Resolve prediction-file paths  # 03.26.2026 Support the new reload-only mode while keeping the prediction-path resolution path unchanged for fresh analysis runs.
# Build the exact .npz file path for each selected CNN run and confirm that every needed prediction file exists when fresh analysis is enabled.  # 03.26.2026 Explain the updated purpose of this cell.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip prediction-path resolution when the notebook is set to reload saved outputs only.
    PREDICTION_PATHS_BY_GROUP = {}  # 03.26.2026 Use an empty placeholder prediction-path dictionary in reload-only mode.
    print("Load-saved-results-only is enabled, so Cell 6 skipped prediction-path resolution.")  # 03.26.2026 Confirm that prediction-path resolution was skipped intentionally.
else:  # 03.26.2026 Run the normal prediction-path resolution path when fresh analysis is enabled.
    if "CNN_RUNS_DF" not in globals():  # 03.26.2026 Require the selected-run table from Cell 5 before resolving prediction files.
        raise RuntimeError("Please run Cell 5 before running this cell.")  # 03.26.2026 Explain which earlier cell provides the needed run table.
    if EVAL_CHOICE == "test_best":  # 03.26.2026 Use the best-checkpoint prediction filename when test_best is selected.
        CHOSEN_PREDS_FILENAME = "test_best_preds.npz"  # 03.26.2026 Keep the saved prediction filename that matches test_best.
    elif EVAL_CHOICE == "test_final":  # 03.26.2026 Use the final-checkpoint prediction filename when test_final is selected.
        CHOSEN_PREDS_FILENAME = "test_final_preds.npz"  # 03.26.2026 Keep the saved prediction filename that matches test_final.
    else:  # 03.26.2026 Stop if the evaluation choice is somehow invalid.
        raise ValueError(f"Unsupported evaluation choice: {EVAL_CHOICE}")  # 03.26.2026 Show the unexpected evaluation choice clearly.
    CNN_RUNS_DF = CNN_RUNS_DF.copy()  # 03.26.2026 Make a safe copy before adding the resolved prediction-path columns.
    CNN_RUNS_DF["chosen_test_artifacts_dir"] = CNN_RUNS_DF["chosen_test_artifacts_dir"].astype(str).str.strip()  # 03.26.2026 Clean the chosen artifact-directory strings.
    CNN_RUNS_DF["chosen_test_artifacts_path"] = CNN_RUNS_DF["chosen_test_artifacts_dir"].apply(Path)  # 03.26.2026 Convert each chosen artifact-directory string into a Path object.
    CNN_RUNS_DF["chosen_preds_filename"] = CHOSEN_PREDS_FILENAME  # 03.26.2026 Save the chosen prediction filename in one shared column.
    CNN_RUNS_DF["chosen_preds_path"] = CNN_RUNS_DF["chosen_test_artifacts_path"].apply(lambda path_value: path_value / CHOSEN_PREDS_FILENAME)  # 03.26.2026 Build the full .npz prediction path for each selected run.
    CNN_RUNS_DF["chosen_preds_exists"] = CNN_RUNS_DF["chosen_preds_path"].apply(lambda path_value: path_value.exists())  # 03.26.2026 Check whether each expected .npz file exists on disk.
    missing_prediction_rows = CNN_RUNS_DF.loc[~CNN_RUNS_DF["chosen_preds_exists"]].copy()  # 03.26.2026 Keep only the runs whose expected .npz file was not found.
    if len(missing_prediction_rows) > 0:  # 03.26.2026 Stop immediately if any expected prediction file is missing.
        missing_messages = []  # 03.26.2026 Collect one readable error line per missing prediction file.
        for _, missing_row in missing_prediction_rows.iterrows():  # 03.26.2026 Walk through each missing prediction row once.
            missing_messages.append(f"{missing_row['group']} | split_seed={missing_row['split_seed']} | trial={missing_row['trial']} | path={missing_row['chosen_preds_path']}")  # 03.26.2026 Save one readable description of the missing prediction file.
        raise FileNotFoundError("One or more selected prediction files were not found:\n" + "\n".join(missing_messages))  # 03.26.2026 Explain exactly which prediction paths are missing.
    CNN_RUNS_DF = CNN_RUNS_DF.sort_values(["group", "split_seed", "trial"]).reset_index(drop=True)  # 03.26.2026 Keep the run table in stable readable order after path resolution.
    PREDICTION_PATHS_BY_GROUP = {}  # 03.26.2026 Collect one selected prediction path per group because this notebook compares one run per group.
    prediction_path_rows = []  # 03.26.2026 Collect short preview rows for the resolved prediction-path table.
    for group_name in SELECTED_GROUPS:  # 03.26.2026 Resolve one selected prediction path for each chosen group.
        group_rows = CNN_RUNS_DF.loc[CNN_RUNS_DF["group"].astype(str) == str(group_name)].copy()  # 03.26.2026 Keep only the run-summary rows for this one group.
        if len(group_rows) == 0:  # 03.26.2026 Stop if the selected group somehow has no matching run rows.
            raise RuntimeError(f"No run rows were found for group: {group_name}")  # 03.26.2026 Explain which group is missing.
        selected_row = group_rows.sort_values(["chosen_test_acc", "best_val_acc", "split_seed", "trial"], ascending=[False, False, True, True], kind="mergesort").iloc[0].copy()  # 03.26.2026 Choose the best available run for this group using the existing stable sort rule.
        PREDICTION_PATHS_BY_GROUP[str(group_name)] = Path(selected_row["chosen_preds_path"])  # 03.26.2026 Save the selected prediction path for this group.
        prediction_path_rows.append({"group": str(group_name), "split_seed": int(selected_row["split_seed"]), "trial": int(selected_row["trial"]), "chosen_test_acc": float(selected_row["chosen_test_acc"]), "chosen_preds_path": str(selected_row["chosen_preds_path"])})  # 03.26.2026 Save one readable preview row for this selected prediction file.
    PREDICTION_PATHS_SUMMARY_DF = pd.DataFrame(prediction_path_rows).sort_values("group").reset_index(drop=True)  # 03.26.2026 Build the one-row-per-group selected prediction-path summary table.
    print("Prediction-file paths resolved successfully.")  # 03.26.2026 Confirm that the selected prediction paths were resolved.
    display(PREDICTION_PATHS_SUMMARY_DF)  # 03.26.2026 Show the selected prediction paths in a stable readable order.


In [ ]:
# Cell 7 — Load CNN predictions  # 03.26.2026 Support the new reload-only mode while keeping the prediction-loading path unchanged for fresh analysis runs.
# Read the selected .npz prediction files, standardize their contents, and store one clean prediction table for each group when fresh analysis is enabled.  # 03.26.2026 Explain the updated purpose of this cell.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip fresh prediction loading when the notebook is set to reload saved outputs only.
    PREDICTIONS_BY_GROUP = {}  # 03.26.2026 Use an empty placeholder raw-prediction dictionary in reload-only mode.
    PREDICTION_TABLES_BY_GROUP = {}  # 03.26.2026 Use an empty placeholder per-group prediction-table dictionary in reload-only mode.
    ALL_PREDICTIONS_LONG_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder long prediction table in reload-only mode.
    print("Load-saved-results-only is enabled, so Cell 7 skipped prediction loading.")  # 03.26.2026 Confirm that prediction loading was skipped intentionally.
else:  # 03.26.2026 Run the normal prediction-loading path when fresh analysis is enabled.
    if "PREDICTION_PATHS_BY_GROUP" not in globals():  # 03.26.2026 Require the resolved prediction paths from Cell 6 before loading .npz files.
        raise RuntimeError("Please run Cell 6 before running this cell.")  # 03.26.2026 Explain which earlier cell provides the needed prediction paths.
    def normalize_class_name(value):  # 03.26.2026 Keep the small helper that converts one saved class-order entry into clean text.
        if isinstance(value, bytes):  # 03.26.2026 Decode bytes when the saved class-order entry is stored as bytes.
            value = value.decode("utf-8")  # 03.26.2026 Convert the bytes object into regular text.
        return str(value).strip()  # 03.26.2026 Return the cleaned class-order entry as plain text.
    PREDICTIONS_BY_GROUP = {}  # 03.26.2026 Collect the raw loaded arrays and summary information for each group.
    PREDICTION_TABLES_BY_GROUP = {}  # 03.26.2026 Collect one cleaned per-image prediction table for each group.
    prediction_preview_rows = []  # 03.26.2026 Collect short preview rows so the loaded prediction files can be inspected together.
    for group_name, prediction_path in PREDICTION_PATHS_BY_GROUP.items():  # 03.26.2026 Load one selected .npz prediction file for each chosen group.
        with np.load(prediction_path, allow_pickle=True) as npz_file:  # 03.26.2026 Open the .npz file safely and allow object arrays if present.
            if "y_true" not in npz_file.files:  # 03.26.2026 Require the true-label array in every saved prediction file.
                raise KeyError(f"'y_true' was not found in {prediction_path}")  # 03.26.2026 Explain which prediction file is missing the true-label array.
            if "y_pred" not in npz_file.files:  # 03.26.2026 Require the predicted-label array in every saved prediction file.
                raise KeyError(f"'y_pred' was not found in {prediction_path}")  # 03.26.2026 Explain which prediction file is missing the predicted-label array.
            if "sids" not in npz_file.files:  # 03.26.2026 Require the stimulus-ID array in every saved prediction file.
                raise KeyError(f"'sids' was not found in {prediction_path}")  # 03.26.2026 Explain which prediction file is missing the stimulus-ID array.
            y_true = np.asarray(npz_file["y_true"]).astype(int)  # 03.26.2026 Load the saved true-label array as integer class IDs.
            y_pred = np.asarray(npz_file["y_pred"]).astype(int)  # 03.26.2026 Load the saved predicted-label array as integer class IDs.
            sids = np.asarray(npz_file["sids"]).astype(int)  # 03.26.2026 Load the saved stimulus-ID array as integers.
            if "class_order" in npz_file.files:  # 03.26.2026 Use the file-specific class order when it was saved inside the .npz file.
                npz_class_order = [normalize_class_name(value) for value in npz_file["class_order"].tolist()]  # 03.26.2026 Read and clean the saved class-order values.
            else:  # 03.26.2026 Fall back to the notebook class order when the .npz file has no class-order array.
                npz_class_order = list(CLASS_ORDER)  # 03.26.2026 Use the shared notebook class order as the fallback.
        if not (len(y_true) == len(y_pred) == len(sids)):  # 03.26.2026 Stop if the core arrays do not describe the same number of test samples.
            raise ValueError(f"Length mismatch in {prediction_path}: len(y_true)={len(y_true)}, len(y_pred)={len(y_pred)}, len(sids)={len(sids)}")  # 03.26.2026 Explain the array-length mismatch clearly.
        if set(npz_class_order) != set(CLASS_ORDER):  # 03.26.2026 Stop if the saved class names do not match the expected six classes.
            raise ValueError(f"Class names in {prediction_path} do not match CLASS_ORDER. Found: {npz_class_order}")  # 03.26.2026 Explain the unexpected saved class names clearly.
        npz_int_to_label = {idx: label for idx, label in enumerate(npz_class_order)}  # 03.26.2026 Build the integer-to-label map for this prediction file's class order.
        true_label_text = pd.Series(y_true).map(npz_int_to_label)  # 03.26.2026 Convert the saved integer true labels into text labels.
        pred_label_text = pd.Series(y_pred).map(npz_int_to_label)  # 03.26.2026 Convert the saved integer predicted labels into text labels.
        if true_label_text.isna().any():  # 03.26.2026 Stop if any saved true-label integer could not be decoded.
            raise ValueError(f"Unmapped true-label integers found in {prediction_path}")  # 03.26.2026 Explain which prediction file contains undecodable true labels.
        if pred_label_text.isna().any():  # 03.26.2026 Stop if any saved predicted-label integer could not be decoded.
            raise ValueError(f"Unmapped predicted-label integers found in {prediction_path}")  # 03.26.2026 Explain which prediction file contains undecodable predicted labels.
        predictions_df = pd.DataFrame({"group": group_name, "stimulus_index": sids, "y_true_int_raw": y_true, "y_pred_int_raw": y_pred, "true_label_raw": true_label_text.astype(str), "pred_label_raw": pred_label_text.astype(str)})  # 03.26.2026 Build one clean per-image prediction table for the current group.
        predictions_df["true_label"] = predictions_df["true_label_raw"].astype(str).str.strip()  # 03.26.2026 Create a cleaned true-label text column.
        predictions_df["pred_label"] = predictions_df["pred_label_raw"].astype(str).str.strip()  # 03.26.2026 Create a cleaned predicted-label text column.
        predictions_df["true_label_int"] = predictions_df["true_label"].map(LABEL_TO_INT)  # 03.26.2026 Convert the cleaned true labels into the notebook integer class IDs.
        predictions_df["pred_label_int"] = predictions_df["pred_label"].map(LABEL_TO_INT)  # 03.26.2026 Convert the cleaned predicted labels into the notebook integer class IDs.
        predictions_df["is_correct"] = predictions_df["true_label"] == predictions_df["pred_label"]  # 03.26.2026 Mark whether each prediction is correct under the original labels.
        if predictions_df["true_label_int"].isna().any():  # 03.26.2026 Stop if any cleaned true label is not in the notebook class order.
            bad_true_labels = sorted(predictions_df.loc[predictions_df["true_label_int"].isna(), "true_label"].unique().tolist())  # 03.26.2026 Collect the unexpected cleaned true labels for the error message.
            raise ValueError(f"Unexpected true-label text found in {prediction_path}: {bad_true_labels}")  # 03.26.2026 Explain which cleaned true labels caused the failure.
        if predictions_df["pred_label_int"].isna().any():  # 03.26.2026 Stop if any cleaned predicted label is not in the notebook class order.
            bad_pred_labels = sorted(predictions_df.loc[predictions_df["pred_label_int"].isna(), "pred_label"].unique().tolist())  # 03.26.2026 Collect the unexpected cleaned predicted labels for the error message.
            raise ValueError(f"Unexpected predicted-label text found in {prediction_path}: {bad_pred_labels}")  # 03.26.2026 Explain which cleaned predicted labels caused the failure.
        predictions_df["true_label_int"] = predictions_df["true_label_int"].astype(int)  # 03.26.2026 Convert the true-label integers into real integer type.
        predictions_df["pred_label_int"] = predictions_df["pred_label_int"].astype(int)  # 03.26.2026 Convert the predicted-label integers into real integer type.
        PREDICTIONS_BY_GROUP[str(group_name)] = {"prediction_path": prediction_path, "class_order_in_file": npz_class_order, "n_samples": len(predictions_df), "y_true": y_true, "y_pred": y_pred, "sids": sids}  # 03.26.2026 Save the raw arrays and summary information for this group.
        PREDICTION_TABLES_BY_GROUP[str(group_name)] = predictions_df.sort_values("stimulus_index").reset_index(drop=True)  # 03.26.2026 Save the cleaned per-image prediction table for this group.
        prediction_preview_rows.append({"group": str(group_name), "prediction_path": str(prediction_path), "n_samples": len(predictions_df), "n_correct": int(predictions_df["is_correct"].sum()), "accuracy_from_npz": float(predictions_df["is_correct"].mean()), "class_order_in_file": ", ".join(npz_class_order)})  # 03.26.2026 Save one readable preview row for this loaded prediction file.
    PREDICTION_LOAD_SUMMARY_DF = pd.DataFrame(prediction_preview_rows).sort_values("group").reset_index(drop=True)  # 03.26.2026 Build the one-row-per-group prediction-load summary table.
    print("Prediction files loaded successfully.")  # 03.26.2026 Confirm that the selected prediction files were read successfully.
    print("Groups loaded:", sorted(PREDICTIONS_BY_GROUP.keys()))  # 03.26.2026 Show which groups now have loaded prediction files.
    display(PREDICTION_LOAD_SUMMARY_DF)  # 03.26.2026 Show the prediction-load summary table inline.


In [ ]:
# Cell 8 — Build the shared misclassification review scope  # 03.26.2026 Support the new reload-only mode while keeping the shared-scope construction path unchanged for fresh analysis runs.
# Combine the loaded CNN predictions with metadata, identify the stimuli misclassified by all selected groups, and prepare that shared set for human review when fresh analysis is enabled.  # 03.26.2026 Explain the updated purpose of this cell.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip shared-scope construction when the notebook is set to reload saved outputs only.
    COMPARISON_WIDE_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder one-row-per-stimulus comparison table in reload-only mode.
    SHARED_MISCLASS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared-misclassification table in reload-only mode.
    REVIEW_SCOPE_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder review-scope table in reload-only mode.
    REVIEW_SCOPE_NAME = ""  # 03.26.2026 Use a blank placeholder review-scope name in reload-only mode.
    GROUP_SUMMARY_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder group summary table in reload-only mode.
    print("Load-saved-results-only is enabled, so Cell 8 skipped shared-scope construction.")  # 03.26.2026 Confirm that shared-scope construction was skipped intentionally.
else:  # 03.26.2026 Run the normal shared-scope construction path when fresh analysis is enabled.
    if "PREDICTION_TABLES_BY_GROUP" not in globals():  # 03.26.2026 Require the loaded per-group prediction tables from Cell 7 before building the shared scope.
        raise RuntimeError("Please run Cell 7 before running this cell.")  # 03.26.2026 Explain which earlier cell provides the needed prediction tables.
    if len(SELECTED_GROUPS) == 0:  # 03.26.2026 Require at least one selected CNN group before building the shared scope.
        raise ValueError("No groups were selected.")  # 03.26.2026 Explain the missing-group problem clearly.
    test_order_df = TEST_SPLIT_DF[["stimulus_index", "position_in_subset"]].copy()  # 03.26.2026 Keep only the test stimulus ID and its order within the CNN test split.
    test_order_df = test_order_df.rename(columns={"position_in_subset": "test_pos"})  # 03.26.2026 Rename the test-order column to a shorter name.
    test_order_df["stimulus_index"] = pd.to_numeric(test_order_df["stimulus_index"], errors="raise").astype(int)  # 03.26.2026 Force stimulus_index to integer type for safe merges.
    test_order_df["test_pos"] = pd.to_numeric(test_order_df["test_pos"], errors="raise").astype(int)  # 03.26.2026 Force test_pos to integer type for safe sorting.
    metadata_columns_to_merge = META_DF.columns.tolist()  # 03.26.2026 Keep the full metadata column list so all stimulus metadata stay available after merging.
    group_prediction_tables = []  # 03.26.2026 Collect one cleaned per-group prediction table at a time.
    group_summary_rows = []  # 03.26.2026 Collect one short summary row for each selected group.
    for group_name in SELECTED_GROUPS:  # 03.26.2026 Process the selected groups one at a time in the chosen order.
        if str(group_name) not in PREDICTION_TABLES_BY_GROUP:  # 03.26.2026 Stop if predictions were not loaded for a selected group.
            raise KeyError(f"Predictions were not loaded for group: {group_name}")  # 03.26.2026 Name the missing selected group clearly.
        group_df = PREDICTION_TABLES_BY_GROUP[str(group_name)].copy()  # 03.26.2026 Read the cleaned per-image prediction table for this selected group.
        group_df["stimulus_index"] = pd.to_numeric(group_df["stimulus_index"], errors="raise").astype(int)  # 03.26.2026 Force stimulus_index to integer type for safe merges.
        group_df = group_df.merge(test_order_df, on="stimulus_index", how="inner", validate="one_to_one")  # 03.26.2026 Attach the saved CNN test position to each prediction row.
        group_df = group_df.merge(META_DF[metadata_columns_to_merge], on="stimulus_index", how="inner", validate="one_to_one")  # 03.26.2026 Attach the full metadata row for each stimulus.
        group_df["stim_ID"] = group_df["stimulus_index"].astype(int)  # 03.26.2026 Keep stim_ID explicit for later shared-scope review files.
        group_df["group"] = str(group_name)  # 03.26.2026 Normalize the group name as plain text on every row.
        if "true_label_x" in group_df.columns and "true_label_y" in group_df.columns:  # 03.26.2026 Handle the expected merge suffixes when both prediction and metadata true-label columns are present.
            group_df["true_label_npz"] = group_df["true_label_x"].astype(str).str.strip()  # 03.26.2026 Preserve the original .npz true label after the merge created suffixes.
            group_df["true_label"] = group_df["true_label_y"].astype(str).str.strip()  # 03.26.2026 Replace the working true label with the metadata true label after the merge.
            group_df = group_df.drop(columns=["true_label_x", "true_label_y"])  # 03.26.2026 Remove the temporary suffixed true-label columns after preserving the needed values.
        else:  # 03.26.2026 Fall back to the unsuffixed true-label column if pandas did not create merge suffixes.
            group_df["true_label_npz"] = group_df["true_label"].astype(str).str.strip()  # 03.26.2026 Preserve the unsuffixed .npz true label when no merge suffixes were created.
            group_df["true_label"] = group_df["true_label"].astype(str).str.strip()  # 03.26.2026 Keep the working true label normalized as plain text when no merge suffixes were created.
        group_df["true_label_int"] = group_df["true_label"].map(LABEL_TO_INT)  # 03.26.2026 Convert the metadata true labels into integer class IDs.
        group_df["npz_true_label"] = group_df["true_label_npz"].astype(str).str.strip()  # 03.26.2026 Keep a cleaned copy of the .npz true label for consistency checks.
        group_df["true_label_matches_metadata"] = group_df["npz_true_label"] == group_df["true_label"]  # 03.26.2026 Record whether the .npz true label agrees with the metadata true label.
        group_df["is_error"] = ~group_df["is_correct"]  # 03.26.2026 Mark whether each prediction is a misclassification.
        if group_df["true_label_int"].isna().any():  # 03.26.2026 Stop if any metadata true label is not part of CLASS_ORDER.
            bad_true_labels = sorted(group_df.loc[group_df["true_label_int"].isna(), "true_label"].unique().tolist())  # 03.26.2026 Collect the unexpected metadata labels for the error message.
            raise ValueError(f"Unexpected metadata true labels found for group {group_name}: {bad_true_labels}")  # 03.26.2026 Explain which metadata labels caused the failure.
        expected_ids = set(TEST_STIMULUS_IDS)  # 03.26.2026 Read the expected final-test stimulus IDs from Cell 5.
        loaded_ids = set(group_df["stimulus_index"].tolist())  # 03.26.2026 Read the actually loaded stimulus IDs for this group.
        missing_ids = sorted(list(expected_ids - loaded_ids))  # 03.26.2026 Find any expected test stimuli that are missing from this group's predictions.
        extra_ids = sorted(list(loaded_ids - expected_ids))  # 03.26.2026 Find any loaded stimuli that are not part of the saved CNN test split.
        if len(missing_ids) > 0 or len(extra_ids) > 0:  # 03.26.2026 Stop if the loaded predictions do not match the saved CNN test split exactly.
            raise ValueError(f"Prediction-test-split mismatch for group {group_name}. Missing IDs: {missing_ids[:20]} | Extra IDs: {extra_ids[:20]}")  # 03.26.2026 Explain the test-split mismatch clearly.
        group_summary_rows.append({"group": str(group_name), "n_test_rows_loaded": len(group_df), "n_errors": int(group_df["is_error"].sum()), "accuracy_from_loaded_predictions": float(group_df["is_correct"].mean()), "all_true_labels_match_metadata": bool(group_df["true_label_matches_metadata"].all())})  # 03.26.2026 Save one readable per-group summary row.
        group_prediction_tables.append(group_df.sort_values("test_pos").reset_index(drop=True))  # 03.26.2026 Keep the cleaned per-group prediction table in saved CNN test order.
    ALL_PREDICTIONS_LONG_DF = pd.concat(group_prediction_tables, ignore_index=True)  # 03.26.2026 Combine all selected groups into one long prediction table.
    ALL_PREDICTIONS_LONG_DF = ALL_PREDICTIONS_LONG_DF.sort_values(["test_pos", "group"]).reset_index(drop=True)  # 03.26.2026 Keep the long prediction table in stable test order and group order.
    base_columns = [column_name for column_name in ["test_pos", "stimulus_index", "stim_ID", "true_label", "true_label_int"] if column_name in ALL_PREDICTIONS_LONG_DF.columns]  # 03.26.2026 Keep the core columns that should appear once in the wide comparison table.
    optional_metadata_columns = [column_name for column_name in ["bg_hue", "bg_int", "obj_int", "delta_int", "bg_sat", "obj_sat", "delta_sat"] if column_name in ALL_PREDICTIONS_LONG_DF.columns]  # 03.26.2026 Keep the same small set of useful metadata columns when they exist.
    base_wide_df = ALL_PREDICTIONS_LONG_DF[base_columns + optional_metadata_columns].drop_duplicates(subset=["stimulus_index"]).copy()  # 03.26.2026 Build the one-row-per-stimulus base table before adding group-specific prediction columns.
    base_wide_df = base_wide_df.sort_values("test_pos").reset_index(drop=True)  # 03.26.2026 Keep the base wide table in saved CNN test order.
    for group_name in SELECTED_GROUPS:  # 03.26.2026 Add one set of group-specific prediction columns for each selected group.
        group_small_df = ALL_PREDICTIONS_LONG_DF.loc[ALL_PREDICTIONS_LONG_DF["group"].astype(str) == str(group_name), ["stimulus_index", "pred_label", "pred_label_int", "is_correct", "is_error"]].copy()  # 03.26.2026 Keep only the key prediction columns needed in the wide table.
        group_small_df = group_small_df.rename(columns={"pred_label": f"pred_label__{group_name}", "pred_label_int": f"pred_label_int__{group_name}", "is_correct": f"is_correct__{group_name}", "is_error": f"is_error__{group_name}"})  # 03.26.2026 Rename the group-specific columns so they stay distinct after merging.
        base_wide_df = base_wide_df.merge(group_small_df, on="stimulus_index", how="left", validate="one_to_one")  # 03.26.2026 Merge this group's prediction columns into the wide comparison table.
    COMPARISON_WIDE_DF = base_wide_df.copy()  # 03.26.2026 Save the completed one-row-per-stimulus comparison table for later cells.
    GROUP_SUMMARY_DF = pd.DataFrame(group_summary_rows).sort_values("group").reset_index(drop=True)  # 03.26.2026 Build the per-group summary table in stable order.
    is_error_columns = [f"is_error__{group_name}" for group_name in SELECTED_GROUPS]  # 03.26.2026 Build the group-specific error-column list in the wide comparison table.
    is_correct_columns = [f"is_correct__{group_name}" for group_name in SELECTED_GROUPS]  # 03.26.2026 Build the group-specific correctness-column list in the wide comparison table.
    for column_name in is_error_columns + is_correct_columns:  # 03.26.2026 Check that all expected correctness and error columns exist.
        if column_name not in COMPARISON_WIDE_DF.columns:  # 03.26.2026 Stop if any expected group-specific column is missing.
            raise KeyError(f"Expected comparison column not found: {column_name}")  # 03.26.2026 Name the missing comparison column clearly.
    COMPARISON_WIDE_DF["n_groups_incorrect"] = COMPARISON_WIDE_DF[is_error_columns].sum(axis=1).astype(int)  # 03.26.2026 Count how many selected groups misclassified each stimulus.
    COMPARISON_WIDE_DF["n_groups_correct"] = COMPARISON_WIDE_DF[is_correct_columns].sum(axis=1).astype(int)  # 03.26.2026 Count how many selected groups classified each stimulus correctly.
    COMPARISON_WIDE_DF["is_shared_misclassification"] = COMPARISON_WIDE_DF["n_groups_incorrect"] == len(SELECTED_GROUPS)  # 03.26.2026 Mark the stimuli misclassified by all selected groups.
    COMPARISON_WIDE_DF["is_any_group_error"] = COMPARISON_WIDE_DF["n_groups_incorrect"] > 0  # 03.26.2026 Mark the stimuli misclassified by at least one selected group.
    SHARED_MISCLASS_DF = COMPARISON_WIDE_DF.loc[COMPARISON_WIDE_DF["is_shared_misclassification"]].copy()  # 03.26.2026 Keep only the shared misclassification set that defines the review scope.
    SHARED_MISCLASS_DF = SHARED_MISCLASS_DF.sort_values("test_pos").reset_index(drop=True)  # 03.26.2026 Keep the shared misclassification set in saved CNN test order.
    SHARED_MISCLASS_DF["review_order"] = np.arange(1, len(SHARED_MISCLASS_DF) + 1)  # 03.26.2026 Add a simple one-based review order for the human-review viewer.
    shared_pred_label_columns = [f"pred_label__{group_name}" for group_name in SELECTED_GROUPS]  # 03.26.2026 Build the predicted-label column list for the selected groups.
    if len(shared_pred_label_columns) > 0:  # 03.26.2026 Build the combined prediction signature when at least one selected group exists.
        SHARED_MISCLASS_DF["shared_pred_label_signature"] = SHARED_MISCLASS_DF[shared_pred_label_columns].astype(str).agg(" | ".join, axis=1)  # 03.26.2026 Create one readable combined prediction signature across the selected groups.
    else:  # 03.26.2026 Handle the unlikely case where no selected predicted-label columns exist.
        SHARED_MISCLASS_DF["shared_pred_label_signature"] = ""  # 03.26.2026 Use a blank signature when no selected predicted-label columns are available.
    REVIEW_SCOPE_NAME = f"shared_misclass_{EVAL_CHOICE}_{'_'.join([str(group_name) for group_name in SELECTED_GROUPS])}"  # 03.26.2026 Build the readable shared review-scope name from the selected evaluation and groups.
    REVIEW_SCOPE_DF = SHARED_MISCLASS_DF.copy()  # 03.26.2026 Save the shared misclassification table again under a review-specific variable name.
    SHARED_CLASS_COUNTS_DF = SHARED_MISCLASS_DF.groupby("true_label", dropna=False).size().reset_index(name="n_shared_misclassified")  # 03.26.2026 Count how many shared misclassifications belong to each true class.
    SHARED_CLASS_COUNTS_DF = SHARED_CLASS_COUNTS_DF.sort_values(["n_shared_misclassified", "true_label"], ascending=[False, True]).reset_index(drop=True)  # 03.26.2026 Put the shared class counts into a stable readable order.
    print("Shared misclassification review scope built successfully.")  # 03.26.2026 Confirm that the shared misclassification set was created.
    print("Selected groups:", SELECTED_GROUPS)  # 03.26.2026 Show which groups were used to define the shared scope.
    print("Total CNN test stimuli:", len(COMPARISON_WIDE_DF))  # 03.26.2026 Show the total size of the full CNN test set.
    print("Shared misclassification stimuli:", len(SHARED_MISCLASS_DF))  # 03.26.2026 Show how many stimuli were misclassified by all selected groups.
    print("Review scope name:", REVIEW_SCOPE_NAME)  # 03.26.2026 Show the generated name for the shared review scope.
    display(GROUP_SUMMARY_DF)  # 03.26.2026 Show one summary row for each selected group.
    display(SHARED_CLASS_COUNTS_DF)  # 03.26.2026 Show how the shared misclassification set is distributed across true classes.
    display(SHARED_MISCLASS_DF.head(20))  # 03.26.2026 Show the first few shared-misclassification rows.


In [ ]:
# Cell 9 — Load and reuse existing human-reviewed labels for the shared misclassification set only  # 03.26.2026 Add mode-aware skipping for automated-label and reload-only runs.
# Read earlier human-label files only when human-review mode is enabled, restrict them to the current shared misclassification set, and identify which shared-scope images still need review.  # 03.26.2026 Explain the updated purpose of this cell.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip reusable-label loading when the notebook is set to reload saved outputs only.
    EXISTING_HUMAN_LABELS_ALL_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder all-label table in reload-only mode.
    EXISTING_HUMAN_LABELS_LATEST_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder latest-label table in reload-only mode.
    EXISTING_HUMAN_LABELS_SHARED_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared-scope label table in reload-only mode.
    SHARED_REVIEW_BASE_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared review-base table in reload-only mode.
    SHARED_REUSED_LABELS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder reused-label table in reload-only mode.
    SHARED_PENDING_REVIEW_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder pending-review table in reload-only mode.
    SHARED_REVIEW_SUMMARY_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared-review summary table in reload-only mode.
    print("Load-saved-results-only is enabled, so Cell 9 skipped reusable human-label loading.")  # 03.26.2026 Confirm that reusable human-label loading was skipped intentionally.
elif not bool(USER_CONFIG["use_human_review"]):  # 03.26.2026 Skip reusable-label loading entirely when automated-label mode is selected.
    EXISTING_HUMAN_LABELS_ALL_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder all-label table in automated-label mode.
    EXISTING_HUMAN_LABELS_LATEST_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder latest-label table in automated-label mode.
    EXISTING_HUMAN_LABELS_SHARED_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared-scope label table in automated-label mode.
    SHARED_REVIEW_BASE_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared review-base table in automated-label mode.
    SHARED_REUSED_LABELS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder reused-label table in automated-label mode.
    SHARED_PENDING_REVIEW_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder pending-review table in automated-label mode.
    SHARED_REVIEW_SUMMARY_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared-review summary table in automated-label mode.
    print("Human review is disabled, so Cell 9 skipped reusable human-label loading.")  # 03.26.2026 Confirm that reusable human-label loading was skipped intentionally.
else:  # 03.26.2026 Run the existing reusable human-label loading path when human-review mode is enabled.
    # Cell 9 — Load and reuse existing human-reviewed labels for the shared misclassification set only
    # Read the earlier saved human-label files, keep only labels for the current shared misclassification set, and identify which shared-set images still need review.

    if "SHARED_MISCLASS_DF" not in globals():  # Make sure Cell 8 was run successfully before loading reusable labels.
        raise RuntimeError("Please run Cell 8 before running this cell.")  # Stop with a clear message if the shared review scope is missing.

    def normalize_existing_human_labels_df(labels_df, source_type, source_name, source_path):  # Define one helper that standardizes older and newer saved human-label tables.
        labels_df = labels_df.copy()  # Work on a copy so the original dataframe is not changed in place.

        if "stim_ID" not in labels_df.columns and "stim_id" in labels_df.columns:  # Check whether an older lowercase stimulus-ID column name was used.
            labels_df = labels_df.rename(columns={"stim_id": "stim_ID"})  # Rename the older stim_id column to the shared stim_ID name.

        required_numeric_columns = ["review_order", "stim_ID", "test_pos", "human_label_int", "previous_cache_hit"]  # List numeric columns that should exist in the standardized table.
        required_text_columns = ["scope_name", "true_label", "human_key", "human_label", "saved_at", "selected_run_keys"]  # List text columns that should exist in the standardized table.

        for column_name in required_numeric_columns:  # Create any missing numeric column one at a time.
            if column_name not in labels_df.columns:  # Check whether this numeric column is missing.
                labels_df[column_name] = np.nan  # Fill the missing numeric column with empty numeric values.

        for column_name in required_text_columns:  # Create any missing text column one at a time.
            if column_name not in labels_df.columns:  # Check whether this text column is missing.
                labels_df[column_name] = ""  # Fill the missing text column with empty strings.

        labels_df["stim_ID"] = pd.to_numeric(labels_df["stim_ID"], errors="coerce")  # Convert stim_ID into numeric values.
        labels_df = labels_df.loc[~labels_df["stim_ID"].isna()].copy()  # Keep only rows with a valid stimulus ID.
        labels_df["stim_ID"] = labels_df["stim_ID"].astype(int)  # Convert stim_ID into integer type.

        labels_df["review_order"] = pd.to_numeric(labels_df["review_order"], errors="coerce")  # Convert review_order into numeric values.
        labels_df["test_pos"] = pd.to_numeric(labels_df["test_pos"], errors="coerce")  # Convert test_pos into numeric values.
        labels_df["human_label_int"] = pd.to_numeric(labels_df["human_label_int"], errors="coerce")  # Convert human_label_int into numeric values when present.
        labels_df["previous_cache_hit"] = pd.to_numeric(labels_df["previous_cache_hit"], errors="coerce").fillna(0).astype(int)  # Convert previous_cache_hit into integer values.

        labels_df["human_label"] = labels_df["human_label"].astype(str).str.strip()  # Clean the saved human-label text values.
        labels_df["true_label"] = labels_df["true_label"].astype(str).str.strip()  # Clean the saved true-label text values.
        labels_df["human_key"] = labels_df["human_key"].astype(str).str.strip()  # Clean the saved keyboard-key text values.
        labels_df["saved_at"] = labels_df["saved_at"].astype(str).str.strip()  # Clean the saved timestamp text values.
        labels_df["scope_name"] = labels_df["scope_name"].astype(str).str.strip()  # Clean the saved scope-name text values.
        labels_df["selected_run_keys"] = labels_df["selected_run_keys"].astype(str).str.strip()  # Clean the saved run-key text values.

        labels_df.loc[labels_df["human_label"].isin(["", "nan", "None"]), "human_label"] = np.nan  # Turn blank-like saved human labels into missing values.
        labels_df.loc[labels_df["true_label"].isin(["", "nan", "None"]), "true_label"] = np.nan  # Turn blank-like saved true labels into missing values.

        labels_df["human_label_int_from_text"] = labels_df["human_label"].map(LABEL_TO_INT)  # Convert the saved human-label text into shared integer class IDs when possible.
        labels_df["true_label_int_from_text"] = labels_df["true_label"].map(LABEL_TO_INT)  # Convert the saved true-label text into shared integer class IDs when possible.
        labels_df["human_label_int"] = labels_df["human_label_int"].fillna(labels_df["human_label_int_from_text"])  # Fill missing human_label_int values from the text label when possible.
        labels_df["true_label_int"] = labels_df["true_label_int_from_text"]  # Create a clean integer version of the saved true-label text.

        labels_df = labels_df.loc[labels_df["human_label"].isin(CLASS_ORDER)].copy()  # Keep only rows whose saved human label is one of the expected class names.
        labels_df["human_label_int"] = labels_df["human_label_int"].astype(int)  # Convert the final human-label integers into integer type.

        labels_df["source_type"] = str(source_type)  # Record whether this row came from the cache or a scope file.
        labels_df["source_name"] = str(source_name)  # Record a short readable source name for this file.
        labels_df["source_path"] = str(source_path)  # Record the full file path for traceability.

        labels_df["saved_at_sort"] = pd.to_datetime(labels_df["saved_at"], errors="coerce")  # Convert saved_at into sortable timestamps when possible.
        labels_df["saved_at_sort"] = labels_df["saved_at_sort"].fillna(pd.Timestamp("1900-01-01"))  # Fill missing timestamps with an old default so real timestamps sort later.

        return labels_df  # Return the standardized human-label dataframe.

    shared_stim_ids = set(SHARED_MISCLASS_DF["stim_ID"].astype(int).tolist())  # Build the set of stimulus IDs that belong to the current shared misclassification review scope.
    existing_label_tables = []  # Collect all existing human-label tables before combining them.
    existing_label_file_rows = []  # Collect one short summary row per loaded human-label file.

    if USER_CONFIG["use_previous_human_labels"]:  # Only load earlier human labels if the user chose to reuse them in Cell 3.
        if EXISTING_CACHE_PATH.exists():  # Check whether the earlier notebook-level cache file exists.
            cache_df_raw = pd.read_csv(EXISTING_CACHE_PATH)  # Read the earlier notebook-level cache CSV.
            cache_df = normalize_existing_human_labels_df(cache_df_raw, source_type="cache", source_name="review_cache_master", source_path=EXISTING_CACHE_PATH)  # Standardize the cache table into the shared schema.
            existing_label_tables.append(cache_df)  # Add the standardized cache rows to the list of existing label tables.
            existing_label_file_rows.append({"source_type": "cache", "source_name": "review_cache_master", "source_path": str(EXISTING_CACHE_PATH), "n_rows_loaded": len(cache_df)})  # Save one short summary row for the cache file.

        scope_label_paths = sorted(EXISTING_SCOPE_DIR.glob("*/scope_human_labels.csv")) if EXISTING_SCOPE_DIR.exists() else []  # Find each saved scope human-label CSV from the earlier review notebook.
        for scope_label_path in scope_label_paths:  # Load each saved scope human-label CSV one at a time.
            scope_df_raw = pd.read_csv(scope_label_path)  # Read the current scope human-label CSV.
            scope_name = scope_label_path.parent.name  # Use the parent folder name as a short readable source name.
            scope_df = normalize_existing_human_labels_df(scope_df_raw, source_type="scope", source_name=scope_name, source_path=scope_label_path)  # Standardize this scope table into the shared schema.
            existing_label_tables.append(scope_df)  # Add the standardized scope rows to the list of existing label tables.
            existing_label_file_rows.append({"source_type": "scope", "source_name": scope_name, "source_path": str(scope_label_path), "n_rows_loaded": len(scope_df)})  # Save one short summary row for this scope file.

    if len(existing_label_tables) == 0:  # Create empty outputs if no earlier reusable labels were found or reuse was turned off.
        EXISTING_HUMAN_LABELS_ALL_DF = pd.DataFrame(columns=["stim_ID", "human_label", "human_label_int", "true_label", "true_label_int", "test_pos", "review_order", "scope_name", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name", "source_path", "saved_at_sort"])  # Create an empty all-labels table with the expected schema.
        EXISTING_HUMAN_LABELS_LATEST_DF = EXISTING_HUMAN_LABELS_ALL_DF.copy()  # Create an empty latest-labels table with the same schema.
        EXISTING_HUMAN_LABELS_FILE_SUMMARY_DF = pd.DataFrame(existing_label_file_rows)  # Create an empty-or-nearly-empty file summary table.
    else:  # Combine, sort, and deduplicate the earlier reusable labels.
        EXISTING_HUMAN_LABELS_ALL_DF = pd.concat(existing_label_tables, ignore_index=True)  # Combine all loaded cache and scope rows into one long table.
        EXISTING_HUMAN_LABELS_ALL_DF = EXISTING_HUMAN_LABELS_ALL_DF.sort_values(["stim_ID", "saved_at_sort", "review_order"], kind="mergesort").reset_index(drop=True)  # Sort repeated rows so the newest row for each stimulus comes last.
        EXISTING_HUMAN_LABELS_LATEST_DF = EXISTING_HUMAN_LABELS_ALL_DF.drop_duplicates(subset=["stim_ID"], keep="last").copy()  # Keep only the newest saved human label for each stimulus ID.
        EXISTING_HUMAN_LABELS_FILE_SUMMARY_DF = pd.DataFrame(existing_label_file_rows).sort_values(["source_type", "source_name"]).reset_index(drop=True)  # Build one short file-summary table.

    if len(EXISTING_HUMAN_LABELS_LATEST_DF) > 0:  # Check whether any latest reusable labels were found.
        EXISTING_HUMAN_LABELS_SHARED_DF = EXISTING_HUMAN_LABELS_LATEST_DF.loc[EXISTING_HUMAN_LABELS_LATEST_DF["stim_ID"].isin(shared_stim_ids)].copy()  # Keep only labels whose stimulus IDs are in the current shared misclassification scope.
    else:  # Create an empty shared-scope labels table if no reusable labels were found.
        EXISTING_HUMAN_LABELS_SHARED_DF = EXISTING_HUMAN_LABELS_LATEST_DF.copy()  # Use the empty latest-labels table directly.

    EXISTING_HUMAN_LABELS_SHARED_DF = EXISTING_HUMAN_LABELS_SHARED_DF.sort_values(["test_pos", "stim_ID"]).reset_index(drop=True)  # Sort the shared-scope reusable labels into a stable readable order.

    SHARED_REVIEW_BASE_DF = SHARED_MISCLASS_DF.copy()  # Copy the current shared misclassification review scope so reusable labels can be merged into it.
    SHARED_REVIEW_BASE_DF["stim_ID"] = pd.to_numeric(SHARED_REVIEW_BASE_DF["stim_ID"], errors="raise").astype(int)  # Force stim_ID to integer type for safe matching to reusable labels.

    reuse_columns = ["stim_ID", "human_label", "human_label_int", "human_key", "saved_at", "source_type", "source_name", "source_path"]  # Keep the key reusable-label columns needed for the current shared review scope.
    reuse_columns = [column_name for column_name in reuse_columns if column_name in EXISTING_HUMAN_LABELS_SHARED_DF.columns]  # Keep only reusable-label columns that actually exist in the earlier files.
    SHARED_REVIEW_BASE_DF = SHARED_REVIEW_BASE_DF.merge(EXISTING_HUMAN_LABELS_SHARED_DF[reuse_columns], on="stim_ID", how="left", validate="one_to_one")  # Merge reusable labels into the current shared misclassification review scope.

    SHARED_REVIEW_BASE_DF["has_reused_human_label"] = SHARED_REVIEW_BASE_DF["human_label"].notna()  # Mark which shared-scope stimuli already have a reusable human-reviewed label.
    SHARED_REVIEW_BASE_DF["human_label_source"] = np.where(SHARED_REVIEW_BASE_DF["has_reused_human_label"], "reused_existing_label", "")  # Record the source of the human label for each shared-scope row.
    SHARED_REVIEW_BASE_DF["needs_new_human_review"] = ~SHARED_REVIEW_BASE_DF["has_reused_human_label"]  # Mark which shared-scope stimuli still need a new manual review.

    SHARED_REUSED_LABELS_DF = SHARED_REVIEW_BASE_DF.loc[SHARED_REVIEW_BASE_DF["has_reused_human_label"]].copy()  # Keep only the shared-scope rows that already have reusable human-reviewed labels.
    SHARED_REUSED_LABELS_DF = SHARED_REUSED_LABELS_DF.sort_values("review_order").reset_index(drop=True)  # Keep the reusable shared-scope labels in review order.

    SHARED_PENDING_REVIEW_DF = SHARED_REVIEW_BASE_DF.loc[SHARED_REVIEW_BASE_DF["needs_new_human_review"]].copy()  # Keep only the shared-scope rows that still need manual review.
    SHARED_PENDING_REVIEW_DF = SHARED_PENDING_REVIEW_DF.sort_values("review_order").reset_index(drop=True)  # Keep the pending shared-scope review rows in review order.

    SHARED_REVIEW_SUMMARY_DF = pd.DataFrame([  # Build one short summary row that describes reusable labels within the current shared scope.
        {
            "review_scope_name": REVIEW_SCOPE_NAME,  # Record the current review-scope name.
            "n_shared_scope_rows": len(SHARED_REVIEW_BASE_DF),  # Record the number of stimuli in the current shared misclassification scope.
            "n_reused_human_labels_in_shared_scope": len(SHARED_REUSED_LABELS_DF),  # Record how many shared-scope rows already have reusable human labels.
            "n_pending_new_reviews_in_shared_scope": len(SHARED_PENDING_REVIEW_DF),  # Record how many shared-scope rows still need new manual review.
            "use_previous_human_labels_setting": bool(USER_CONFIG["use_previous_human_labels"]),  # Record whether label reuse was enabled.
        }
    ])  # Finish the shared-scope review summary table.

    print("Existing human-reviewed labels were loaded and restricted to the shared misclassification set.")  # Confirm that reusable labels were processed for the current shared scope only.
    display(SHARED_REVIEW_SUMMARY_DF)  # Show the one-row summary of reusable labels and pending reviews for the current shared scope.
    display(EXISTING_HUMAN_LABELS_FILE_SUMMARY_DF.head(20))  # Show a short summary of the earlier cache and scope label files that were read.
    display(SHARED_REUSED_LABELS_DF[["review_order", "test_pos", "stim_ID", "true_label", "human_label", "source_type", "source_name", "saved_at"]].head(20))  # Show the first few shared-scope rows that already have reusable human-reviewed labels.
    display(SHARED_PENDING_REVIEW_DF[["review_order", "test_pos", "stim_ID", "true_label", "shared_pred_label_signature"]].head(20))  # Show the first few shared-scope rows that still need new manual review.


In [ ]:
# Cell 10 — Interactive human-review viewer for the shared misclassification scope  # 03.26.2026 Add mode-aware skipping for automated-label and reload-only runs.
# Review the shared-scope images one by one only when human-review mode is enabled and fresh analysis is active.  # 03.26.2026 Explain the updated purpose of this cell.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip the interactive viewer when the notebook is set to reload saved outputs only.
    print("Load-saved-results-only is enabled, so Cell 10 skipped the interactive human-review viewer.")  # 03.26.2026 Confirm that the interactive viewer was skipped intentionally.
elif not bool(USER_CONFIG["use_human_review"]):  # 03.26.2026 Skip the interactive viewer entirely when automated-label mode is selected.
    print("Human review is disabled, so Cell 10 skipped the interactive human-review viewer.")  # 03.26.2026 Confirm that the interactive viewer was skipped intentionally.
else:  # 03.26.2026 Run the existing interactive viewer path when human-review mode is enabled.
    # Cell 10 — Interactive human-review viewer for the shared misclassification scope
    # Review the shared-scope images one by one, type one key in the text box, press Enter to save immediately, and save new labels right away.

    if "REVIEW_SCOPE_NAME" not in globals() or "SHARED_REVIEW_BASE_DF" not in globals():  # Stop if the shared review scope has not been built yet.
        raise RuntimeError("Please run Cells 8 and 9 before running this cell.")  # Explain which earlier cells are required first.

    NOTEBOOK_OUTPUT_DIR = USER_CONFIG["output_dir"]  # Read the notebook output directory chosen in Cell 3.
    NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Create the notebook output folder if it does not already exist.

    CURRENT_SCOPE_DIR = NOTEBOOK_OUTPUT_DIR / "review_scopes" / REVIEW_SCOPE_NAME  # Create one folder for the current shared-misclassification review scope.
    CURRENT_SCOPE_DIR.mkdir(parents=True, exist_ok=True)  # Create the current scope folder if it does not already exist.

    CURRENT_SCOPE_NEW_LABELS_PATH = CURRENT_SCOPE_DIR / "scope_human_labels_new_only.csv"  # Save only the new labels created in this notebook run here.
    CURRENT_SCOPE_HUMAN_LABELS_PATH = CURRENT_SCOPE_DIR / "scope_human_labels.csv"  # Save the combined current-scope labels here.
    CURRENT_SCOPE_SESSION_INFO_PATH = CURRENT_SCOPE_DIR / "session_info.json"  # Save a small JSON file describing this review session.

    NOTEBOOK_CACHE_DIR = NOTEBOOK_OUTPUT_DIR / "cache"  # Create a cache folder for this new notebook's reusable human labels.
    NOTEBOOK_CACHE_DIR.mkdir(parents=True, exist_ok=True)  # Create the cache folder if it does not already exist.
    NOTEBOOK_CACHE_PATH = NOTEBOOK_CACHE_DIR / "review_cache_master.csv"  # Save this notebook's reusable cache here.

    KEY_HELP_TEXT = "a=red, s=green, d=blue, f=yellow, k=dark gray, l=light gray"  # Build one short help string that explains the valid review keys.

    session_info = {  # Collect a few readable settings so the current scope folder is self-documented.
        "review_scope_name": REVIEW_SCOPE_NAME,  # Record the current shared-misclassification review-scope name.
        "eval_choice": USER_CONFIG["eval_choice"],  # Record whether the notebook is using test_best or test_final predictions.
        "groups": USER_CONFIG["groups"],  # Record the selected CNN groups used to define this shared scope.
        "output_dir": str(NOTEBOOK_OUTPUT_DIR),  # Record the full notebook output directory as text.
        "scope_dir": str(CURRENT_SCOPE_DIR),  # Record the full current-scope output directory as text.
        "created_at": datetime.now().isoformat(timespec="seconds"),  # Record when this session-info file was created.
    }  # Finish the session-info dictionary.

    with open(CURRENT_SCOPE_SESSION_INFO_PATH, "w") as file_handle:  # Open the session-info JSON file for writing.
        json.dump(session_info, file_handle, indent=2)  # Save the session-info dictionary in a readable JSON format.

    def open_image_zarr_array(zarr_path):  # Define one helper that opens dataset.zarr and finds the image array inside it.
        zarr_object = zarr.open(str(zarr_path), mode="r")  # Open the zarr dataset in read-only mode.

        if isinstance(zarr_object, zarr.Array):  # Check whether the zarr root is already an array.
            return zarr_object  # Return it directly if the root itself is the image array.

        candidate_names = ["imgs", "images", "stimuli", "X", "data"]  # List a few common image-array names to check first.
        for candidate_name in candidate_names:  # Check each common image-array name one at a time.
            if candidate_name in zarr_object:  # See whether this candidate name exists in the zarr group.
                candidate_object = zarr_object[candidate_name]  # Read the object stored at that candidate name.
                if isinstance(candidate_object, zarr.Array):  # Check whether the candidate object is actually an array.
                    return candidate_object  # Return the first matching array immediately.

        for array_name in zarr_object.array_keys():  # Fall back to the first direct child array if no common name matched.
            return zarr_object[array_name]  # Return the first child array found in the zarr group.

        raise ValueError(f"Could not find an image array inside: {zarr_path}")  # Stop with a clear error if no usable image array was found.

    def prepare_image_for_display(image_array, stimulus_index):  # Define one helper that reads and formats one stimulus image for matplotlib display.
        image_data = np.asarray(image_array[int(stimulus_index)])  # Read one stimulus image from the zarr array using its integer stimulus index.

        if image_data.ndim == 2:  # Check whether the image is already a 2D grayscale image.
            return image_data  # Return the 2D grayscale image directly.

        if image_data.ndim == 3 and image_data.shape[0] in [1, 3, 4] and image_data.shape[-1] not in [1, 3, 4]:  # Check whether the image uses channel-first layout.
            image_data = np.moveaxis(image_data, 0, -1)  # Move the channel axis to the end so matplotlib can display it correctly.

        if image_data.ndim == 3 and image_data.shape[-1] == 1:  # Check whether the image has one trailing channel.
            image_data = image_data[..., 0]  # Remove the single trailing channel for simpler grayscale display.

        return image_data  # Return the display-ready image array.

    IMAGE_ARRAY = open_image_zarr_array(ZARR_PATH)  # Open the shared stimulus-image dataset and keep the image array ready for review.

    def normalize_scope_labels_df(labels_df):  # Standardize the current scope's combined human-label table to one shared schema.
        labels_df = labels_df.copy()  # Work on a copy so the caller's dataframe is not modified in place.

        if "stim_ID" not in labels_df.columns and "stim_id" in labels_df.columns:  # Rename lowercase stim_id if that older name was used.
            labels_df = labels_df.rename(columns={"stim_id": "stim_ID"})  # Rename the lowercase stim_id column to the shared stim_ID name.

        required_text_cols = ["scope_name", "true_label", "human_key", "human_label", "saved_at", "selected_run_keys", "source_type", "source_name"]  # List the expected text columns.
        required_numeric_cols = ["review_order", "stim_ID", "test_pos", "stimulus_index", "human_label_int", "previous_cache_hit"]  # List the expected numeric columns.

        for column_name in required_text_cols:  # Create any missing text column one at a time.
            if column_name not in labels_df.columns:  # Check whether this text column is missing.
                labels_df[column_name] = ""  # Fill the missing text column with empty strings.

        for column_name in required_numeric_cols:  # Create any missing numeric column one at a time.
            if column_name not in labels_df.columns:  # Check whether this numeric column is missing.
                labels_df[column_name] = np.nan  # Fill the missing numeric column with NaN values.

        labels_df["stim_ID"] = pd.to_numeric(labels_df["stim_ID"], errors="coerce")  # Convert stim_ID into numeric values.
        labels_df["stimulus_index"] = pd.to_numeric(labels_df["stimulus_index"], errors="coerce")  # Convert stimulus_index into numeric values.
        labels_df["test_pos"] = pd.to_numeric(labels_df["test_pos"], errors="coerce")  # Convert test_pos into numeric values.
        labels_df["review_order"] = pd.to_numeric(labels_df["review_order"], errors="coerce")  # Convert review_order into numeric values.
        labels_df["human_label_int"] = pd.to_numeric(labels_df["human_label_int"], errors="coerce")  # Convert human_label_int into numeric values.
        labels_df["previous_cache_hit"] = pd.to_numeric(labels_df["previous_cache_hit"], errors="coerce").fillna(0).astype(int)  # Convert previous_cache_hit into 0/1 integers.

        labels_df = labels_df.loc[~labels_df["stim_ID"].isna()].copy()  # Keep only rows with a valid stim_ID.
        labels_df["stim_ID"] = labels_df["stim_ID"].astype(int)  # Convert stim_ID into integer type.
        labels_df["stimulus_index"] = labels_df["stimulus_index"].fillna(labels_df["stim_ID"]).astype(int)  # Fill missing stimulus_index values from stim_ID and convert to integer type.

        labels_df["human_label"] = labels_df["human_label"].astype(str).str.strip()  # Clean the saved human-label text values.
        labels_df["true_label"] = labels_df["true_label"].astype(str).str.strip()  # Clean the saved true-label text values.
        labels_df["human_key"] = labels_df["human_key"].astype(str).str.strip().str.lower()  # Clean the saved keyboard-key text values.
        labels_df["saved_at"] = labels_df["saved_at"].astype(str).str.strip()  # Clean the saved timestamp text values.
        labels_df["scope_name"] = labels_df["scope_name"].astype(str).str.strip()  # Clean the saved scope-name text values.
        labels_df["selected_run_keys"] = labels_df["selected_run_keys"].astype(str).str.strip()  # Clean the saved selected-group key text values.
        labels_df["source_type"] = labels_df["source_type"].astype(str).str.strip()  # Clean the saved source-type text values.
        labels_df["source_name"] = labels_df["source_name"].astype(str).str.strip()  # Clean the saved source-name text values.

        labels_df = labels_df.loc[labels_df["human_label"].isin(CLASS_ORDER)].copy()  # Keep only rows whose human label is one of the six expected class names.
        labels_df["human_label_int"] = labels_df["human_label"].map(LABEL_TO_INT).astype(int)  # Recompute the integer human-label value from the cleaned text label.

        labels_df["saved_at_sort"] = pd.to_datetime(labels_df["saved_at"], errors="coerce")  # Convert saved_at into sortable timestamps.
        labels_df["saved_at_sort"] = labels_df["saved_at_sort"].fillna(pd.Timestamp("1900-01-01"))  # Fill missing timestamps with an old default date.
        labels_df = labels_df.sort_values(["stim_ID", "saved_at_sort", "review_order"], kind="mergesort")  # Put repeated rows into a stable order.
        labels_df = labels_df.drop_duplicates(subset=["stim_ID"], keep="last").reset_index(drop=True)  # Keep only the newest saved row for each stim_ID.

        return labels_df  # Return the standardized scope-label dataframe.

    def build_reused_scope_labels_df():  # Build a standardized dataframe of the reused existing labels for the current shared scope.
        if len(SHARED_REUSED_LABELS_DF) == 0:  # Return an empty standardized dataframe when no shared-scope reused labels exist.
            return normalize_scope_labels_df(pd.DataFrame(columns=["review_order", "scope_name", "stim_ID", "stimulus_index", "test_pos", "true_label", "human_key", "human_label", "human_label_int", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name"]))  # Return the expected empty schema.

        reused_df = SHARED_REUSED_LABELS_DF.copy()  # Copy the shared reused-label rows so they can be renamed and standardized safely.
        reused_df["scope_name"] = REVIEW_SCOPE_NAME  # Record the current shared-scope name on each reused row.
        reused_df["stimulus_index"] = pd.to_numeric(reused_df["stimulus_index"], errors="coerce").fillna(reused_df["stim_ID"]).astype(int)  # Make sure a usable stimulus_index column exists.
        reused_df["previous_cache_hit"] = 1  # Mark these rows as reused earlier human labels.
        reused_df["selected_run_keys"] = "|".join([str(group_name) for group_name in SELECTED_GROUPS])  # Record the selected CNN groups that define this shared scope.
        reused_df["source_type"] = reused_df["source_type"].astype(str) if "source_type" in reused_df.columns else "reused_existing_label"  # Keep any existing source_type value or use a default.
        reused_df["source_name"] = reused_df["source_name"].astype(str) if "source_name" in reused_df.columns else "reused_existing_label"  # Keep any existing source_name value or use a default.

        keep_columns = ["review_order", "scope_name", "stim_ID", "stimulus_index", "test_pos", "true_label", "human_key", "human_label", "human_label_int", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name"]  # Keep the columns needed for the standardized current-scope label table.
        reused_df = reused_df[keep_columns].copy()  # Keep only the standardized current-scope label columns.
        reused_df = normalize_scope_labels_df(reused_df)  # Standardize the reused-label dataframe.

        return reused_df  # Return the standardized reused current-scope label dataframe.

    def normalize_new_scope_labels_df(labels_df):  # Standardize the new labels created in this notebook run.
        labels_df = labels_df.copy()  # Work on a copy so the caller's dataframe is not changed in place.

        required_columns = ["review_order", "scope_name", "stim_ID", "stimulus_index", "test_pos", "true_label", "human_key", "human_label", "human_label_int", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name"]  # List the columns expected in the new-only label CSV.
        for column_name in required_columns:  # Create any missing expected column one at a time.
            if column_name not in labels_df.columns:  # Check whether the expected column is missing.
                labels_df[column_name] = np.nan if column_name in ["review_order", "stim_ID", "stimulus_index", "test_pos", "human_label_int", "previous_cache_hit"] else ""  # Fill the missing column with a simple default value.

        labels_df = normalize_scope_labels_df(labels_df)  # Reuse the general scope-label standardization helper.
        labels_df["source_type"] = "new_review"  # Mark these rows as newly created labels in this notebook run.
        labels_df["source_name"] = REVIEW_SCOPE_NAME  # Mark the current review scope as the source name for these new rows.

        return labels_df  # Return the standardized new-only label dataframe.

    def load_new_scope_labels_df():  # Load the new labels created in this notebook run for the current shared scope.
        if not CURRENT_SCOPE_NEW_LABELS_PATH.exists():  # Create the expected empty new-only label table if it does not exist yet.
            empty_df = pd.DataFrame(columns=["review_order", "scope_name", "stim_ID", "stimulus_index", "test_pos", "true_label", "human_key", "human_label", "human_label_int", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name"])  # Define the empty new-only label schema.
            empty_df.to_csv(CURRENT_SCOPE_NEW_LABELS_PATH, index=False)  # Save the empty new-only label CSV immediately.
            return empty_df  # Return the empty new-only label dataframe.
        loaded_df = pd.read_csv(CURRENT_SCOPE_NEW_LABELS_PATH)  # Load the existing new-only label CSV for this scope.
        return normalize_new_scope_labels_df(loaded_df)  # Standardize and return the loaded new-only label dataframe.

    def save_new_scope_labels_df(labels_df):  # Save the current new-only scope label dataframe back to disk.
        save_df = normalize_new_scope_labels_df(labels_df) if len(labels_df) > 0 else labels_df.copy()  # Standardize the dataframe before saving when it contains rows.
        save_df.to_csv(CURRENT_SCOPE_NEW_LABELS_PATH, index=False)  # Write the new-only label dataframe to disk immediately.

    def build_combined_scope_labels_df():  # Build the current scope's full label table by combining reused and newly created labels.
        reused_df = build_reused_scope_labels_df()  # Build the standardized reused-label dataframe for this shared scope.
        new_df = load_new_scope_labels_df()  # Load the standardized new-only label dataframe for this shared scope.
        combined_df = pd.concat([reused_df, new_df], ignore_index=True) if len(reused_df) > 0 or len(new_df) > 0 else pd.DataFrame(columns=["review_order", "scope_name", "stim_ID", "stimulus_index", "test_pos", "true_label", "human_key", "human_label", "human_label_int", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name"])  # Combine reused and new labels into one current-scope dataframe.
        combined_df = normalize_scope_labels_df(combined_df) if len(combined_df) > 0 else combined_df  # Standardize the combined scope-label dataframe.
        return combined_df  # Return the combined current-scope label dataframe.

    def save_combined_scope_labels_df():  # Save the combined current-scope human-label table to disk.
        combined_df = build_combined_scope_labels_df()  # Rebuild the full current-scope label table from reused and new labels.
        combined_df.to_csv(CURRENT_SCOPE_HUMAN_LABELS_PATH, index=False)  # Save the combined current-scope label CSV immediately.

    def load_notebook_cache(cache_path):  # Load this new notebook's reusable human-label cache from disk.
        if not cache_path.exists():  # Create the expected empty cache file if it does not exist yet.
            empty_cache_df = pd.DataFrame(columns=["review_order", "scope_name", "stim_id", "stimulus_index", "test_pos", "true_label", "human_key", "human_label", "human_label_int", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name"])  # Define the empty notebook-cache schema.
            empty_cache_df.to_csv(cache_path, index=False)  # Save the empty notebook cache CSV immediately.
            return empty_cache_df  # Return the empty notebook-cache dataframe.
        return pd.read_csv(cache_path)  # Load and return the saved notebook-cache dataframe.

    def append_to_notebook_cache(label_row):  # Append one newly created label row to this notebook's reusable cache.
        cache_df = load_notebook_cache(NOTEBOOK_CACHE_PATH)  # Load the current notebook-cache dataframe from disk.
        new_cache_row = pd.DataFrame([  # Build one new notebook-cache row from the saved current-scope label row.
            {
                "review_order": int(label_row["review_order"]),  # Save the review-order value.
                "scope_name": str(label_row["scope_name"]),  # Save the scope name.
                "stim_id": int(label_row["stim_ID"]),  # Save the stim_ID using the cache file's lowercase stim_id style.
                "stimulus_index": int(label_row["stimulus_index"]),  # Save the stimulus index used to read the image.
                "test_pos": int(label_row["test_pos"]),  # Save the CNN test position.
                "true_label": str(label_row["true_label"]),  # Save the metadata true label.
                "human_key": str(label_row["human_key"]),  # Save the typed review key.
                "human_label": str(label_row["human_label"]),  # Save the human-reviewed label.
                "human_label_int": int(label_row["human_label_int"]),  # Save the integer human-label value.
                "saved_at": str(label_row["saved_at"]),  # Save the timestamp.
                "selected_run_keys": str(label_row["selected_run_keys"]),  # Save the selected-group string.
                "previous_cache_hit": int(label_row["previous_cache_hit"]),  # Save whether this image already had a reused label.
                "source_type": str(label_row["source_type"]),  # Save the source type for traceability.
                "source_name": str(label_row["source_name"]),  # Save the source name for traceability.
            }
        ])  # Finish the new notebook-cache row.
        cache_df = pd.concat([cache_df, new_cache_row], ignore_index=True)  # Append the new notebook-cache row.
        cache_df.to_csv(NOTEBOOK_CACHE_PATH, index=False)  # Save the updated notebook-cache file immediately.

    save_combined_scope_labels_df()  # Make sure the current scope's combined label file exists before the viewer starts.

    def current_scope_sequence_df():  # Build the current visible review sequence for the active review mode.
        scope_df = SHARED_REVIEW_BASE_DF.copy()  # Start from the full shared misclassification review scope built in Cells 8 and 9.
        scope_df["stim_ID"] = pd.to_numeric(scope_df["stim_ID"], errors="coerce")  # Convert stim_ID into numeric values.
        scope_df["stimulus_index"] = pd.to_numeric(scope_df["stimulus_index"], errors="coerce")  # Convert stimulus_index into numeric values.
        scope_df["test_pos"] = pd.to_numeric(scope_df["test_pos"], errors="coerce")  # Convert test_pos into numeric values.
        scope_df = scope_df.loc[~scope_df["stim_ID"].isna()].copy()  # Keep only rows with a valid stim_ID.
        scope_df["stim_ID"] = scope_df["stim_ID"].astype(int)  # Convert stim_ID into integer type.
        scope_df["stimulus_index"] = scope_df["stimulus_index"].fillna(scope_df["stim_ID"]).astype(int)  # Fill missing stimulus_index values from stim_ID and convert to integer type.

        combined_scope_labels_df = build_combined_scope_labels_df()  # Load the current scope's combined saved labels.
        labeled_ids = set(combined_scope_labels_df["stim_ID"].astype(int).tolist()) if len(combined_scope_labels_df) > 0 else set()  # Build the set of stim_ID values that already have a current-scope saved label.

        if review_mode_dropdown.value == "pending":  # Keep only currently unlabeled scope items in pending-only mode.
            scope_df = scope_df.loc[~scope_df["stim_ID"].isin(labeled_ids)].copy()  # Drop any shared-scope items that already have a current-scope saved label.

        scope_df = scope_df.sort_values(["test_pos", "stim_ID"], kind="mergesort").reset_index(drop=True)  # Put the visible review sequence into stable CNN test-set order.
        return scope_df  # Return the currently visible review sequence.

    def refresh_review_slider():  # Reset the slider to match the current visible review-sequence length.
        sequence_df = current_scope_sequence_df()  # Build the visible review sequence.
        if len(sequence_df) == 0:  # Handle the case where no scope items are visible in the current review mode.
            review_index_slider.min = 0  # Keep the slider minimum at zero.
            review_index_slider.max = 0  # Keep the slider maximum at zero.
            review_index_slider.value = 0  # Keep the slider value at zero.
            review_index_slider.disabled = True  # Disable the slider when no review items are visible.
        else:  # Otherwise set the slider range to match the visible review sequence.
            current_value = int(review_index_slider.value) if review_index_slider.value is not None else 0  # Read the current slider value safely.
            review_index_slider.min = 0  # Set the slider minimum to the first visible item.
            review_index_slider.max = int(len(sequence_df) - 1)  # Set the slider maximum to the last visible item.
            review_index_slider.value = min(current_value, int(len(sequence_df) - 1))  # Keep the current slider value if it is still in range.
            review_index_slider.disabled = False  # Enable the slider when at least one review item is visible.

    def current_item_row():  # Read the currently selected review item row from the visible review sequence.
        sequence_df = current_scope_sequence_df()  # Build the visible review sequence.
        if len(sequence_df) == 0:  # Return None when the visible sequence is empty.
            return None  # There is no current review item to display.
        current_index = int(review_index_slider.value)  # Read the current visible-item index from the slider.
        return sequence_df.iloc[current_index].copy()  # Return the current visible review row as a copy.

    def build_prediction_table_for_stim(stim_id):  # Build the selected-group prediction table for one reviewed stimulus.
        stim_id = int(stim_id)  # Convert the incoming stim_ID into an integer.
        predictions_df = ALL_PREDICTIONS_LONG_DF.copy()  # Start from the saved long-format shared prediction table.
        predictions_df["stim_ID"] = pd.to_numeric(predictions_df["stim_ID"], errors="coerce")  # Convert stim_ID into numeric values.
        predictions_df = predictions_df.loc[~predictions_df["stim_ID"].isna()].copy()  # Keep only rows with a valid stim_ID.
        predictions_df["stim_ID"] = predictions_df["stim_ID"].astype(int)  # Convert stim_ID into integer type.
        predictions_df = predictions_df.loc[predictions_df["stim_ID"] == stim_id, ["group", "pred_label", "is_correct"]].copy()  # Keep only the selected-group prediction rows for this one stim_ID.
        predictions_df = predictions_df.sort_values(["group"], kind="mergesort").reset_index(drop=True)  # Put the prediction rows into stable readable order.
        return predictions_df  # Return the selected-group prediction table for this stim_ID.

    def save_current_label_from_key(human_key):  # Save one human label immediately for the current visible review item.
        human_key = str(human_key).strip().lower()  # Normalize the typed key to lowercase text.
        if human_key not in KEY_TO_LABEL:  # Ignore invalid keys that are not in the review key map.
            with status_output:  # Write the invalid-key message in the status area.
                clear_output(wait=True)  # Clear the previous status message first.
                print("Invalid key. Use:", KEY_HELP_TEXT)  # Remind the user of the valid key map.
            return  # Stop this save action here.

        item_row = current_item_row()  # Read the current visible review item.
        if item_row is None:  # Stop if there is no current item to save.
            with status_output:  # Write the no-items message in the status area.
                clear_output(wait=True)  # Clear the previous status message first.
                print("No review items are currently visible.")  # Explain why nothing was saved.
            return  # Stop this save action here.

        new_labels_df = load_new_scope_labels_df()  # Load the current scope's new-only human labels.
        stim_id = int(item_row["stim_ID"])  # Read the current stim_ID as an integer.
        stimulus_index = int(item_row["stimulus_index"])  # Read the current stimulus_index as an integer.
        true_label = str(item_row["true_label"])  # Read the current true label as text.
        test_pos = int(item_row["test_pos"])  # Read the current test position as an integer.
        human_label = str(KEY_TO_LABEL[human_key])  # Convert the typed key into the saved human-label text.
        human_label_int = int(LABEL_TO_INT[human_label])  # Convert the human label into its integer class ID.
        now_text = datetime.now().isoformat(timespec="seconds")  # Build one readable save timestamp string.
        previous_cache_hit_value = int(bool(item_row.get("has_reused_human_label", False))) if "has_reused_human_label" in item_row.index else 0  # Record whether this stimulus already had a reused earlier human label.
        next_review_order = int(new_labels_df["review_order"].max()) + 1 if len(new_labels_df) > 0 and new_labels_df["review_order"].notna().any() else 1  # Build the next review_order value for the new-only file.

        new_labels_df = new_labels_df.loc[new_labels_df["stim_ID"] != stim_id].copy() if len(new_labels_df) > 0 else new_labels_df.copy()  # Remove any older new-only row for this same stim_ID before appending the newest one.

        new_label_row = {  # Build the new saved label row for this current scope.
            "review_order": int(next_review_order),  # Save the new review_order value.
            "scope_name": str(REVIEW_SCOPE_NAME),  # Save the current review-scope name.
            "stim_ID": int(stim_id),  # Save the current stim_ID.
            "stimulus_index": int(stimulus_index),  # Save the current stimulus index used to read the image.
            "test_pos": int(test_pos),  # Save the current CNN test position.
            "true_label": str(true_label),  # Save the metadata true label.
            "human_key": str(human_key),  # Save the typed human-review key.
            "human_label": str(human_label),  # Save the human-reviewed label text.
            "human_label_int": int(human_label_int),  # Save the integer human-label value.
            "saved_at": str(now_text),  # Save the timestamp.
            "selected_run_keys": "|".join([str(group_name) for group_name in SELECTED_GROUPS]),  # Save the selected-group string for traceability.
            "previous_cache_hit": int(previous_cache_hit_value),  # Save whether this stimulus already had a reused earlier label.
            "source_type": "new_review",  # Mark this row as a newly created label in this notebook run.
            "source_name": REVIEW_SCOPE_NAME,  # Mark the current review-scope name as the source name.
        }  # Finish the new saved-label row.

        new_labels_df = pd.concat([new_labels_df, pd.DataFrame([new_label_row])], ignore_index=True)  # Append the new saved row to the current scope's new-only label table.
        new_labels_df = normalize_new_scope_labels_df(new_labels_df)  # Standardize the updated new-only label table.
        save_new_scope_labels_df(new_labels_df)  # Save the updated new-only label CSV immediately.
        save_combined_scope_labels_df()  # Rebuild and save the combined current-scope label CSV immediately.
        append_to_notebook_cache(new_label_row)  # Also save the newest label immediately into this notebook's reusable cache.

        with status_output:  # Write the save confirmation in the status area.
            clear_output(wait=True)  # Clear the previous status message first.
            print(f"Saved label for stim_ID {stim_id}: key='{human_key}' -> {human_label}")  # Confirm exactly what label was saved.

        if review_mode_dropdown.value == "pending":  # Move naturally through the remaining unlabeled shared-scope images in pending-only mode.
            refresh_review_slider()  # Rebuild the visible-sequence slider after one pending item was removed.
        else:  # Otherwise keep the all-items sequence and move to the next visible item when possible.
            if int(review_index_slider.value) < int(review_index_slider.max):  # Move forward only if a later visible item still exists.
                review_index_slider.value = int(review_index_slider.value) + 1  # Advance to the next visible review item.

        render_current_review_item()  # Redraw the current review item immediately after saving.

    def render_current_review_item(*_args):  # Draw the current visible review item, its image, and its label information.
        item_row = current_item_row()  # Read the current visible review item row.
        with review_output:  # Draw everything into the review-output area.
            clear_output(wait=True)  # Clear the previous review display first.

            combined_scope_labels_df = build_combined_scope_labels_df()  # Load the current scope's combined saved labels.
            total_shared = len(SHARED_REVIEW_BASE_DF)  # Count the total number of images in the shared misclassification scope.
            total_combined_labeled = len(combined_scope_labels_df)  # Count how many shared-scope images already have a current-scope saved label.
            total_pending = max(0, total_shared - total_combined_labeled)  # Count how many shared-scope images still need a current-scope label.

            print(f"Current scope: {REVIEW_SCOPE_NAME}")  # Show the current shared review-scope name.
            print(f"Key map: {KEY_HELP_TEXT}")  # Remind the user of the valid key map.
            print(f"Shared-scope images: {total_shared} | Current-scope labeled: {total_combined_labeled} | Pending: {total_pending}")  # Show a short scope progress summary.
            print()  # Add a blank line before the item-specific information.

            if item_row is None:  # Handle the case where no review items are visible in the current mode.
                print("No review items are visible for the current mode.")  # Explain why nothing is shown.
                return  # Stop drawing here.

            stim_id = int(item_row["stim_ID"])  # Read the stim_ID of the current review item.
            stimulus_index = int(item_row["stimulus_index"])  # Read the stimulus_index of the current review item.
            true_label = str(item_row["true_label"])  # Read the true label of the current review item.
            test_pos = int(item_row["test_pos"])  # Read the CNN test position of the current review item.
            image_array = prepare_image_for_display(IMAGE_ARRAY, stimulus_index)  # Read and format the current image from dataset.zarr.

            previous_human_label = str(item_row["human_label"]) if "human_label" in item_row.index and pd.notna(item_row["human_label"]) else ""  # Read any reused earlier human label already attached to this shared-scope row.
            previous_human_saved_at = str(item_row["saved_at"]) if "saved_at" in item_row.index and pd.notna(item_row["saved_at"]) else ""  # Read when that reused earlier human label was saved.
            predictions_df = build_prediction_table_for_stim(stim_id)  # Build the selected-group prediction table for this stim_ID.

            current_scope_label_text = ""  # Start with an empty current-scope saved-label summary.
            if len(combined_scope_labels_df) > 0 and stim_id in set(combined_scope_labels_df["stim_ID"].astype(int).tolist()):  # Show the current-scope saved label when this stim_ID already has one.
                current_scope_row = combined_scope_labels_df.loc[combined_scope_labels_df["stim_ID"].astype(int) == stim_id].iloc[-1]  # Read the newest current-scope saved-label row for this stim_ID.
                current_scope_label_text = f"{str(current_scope_row['human_label'])} (key={str(current_scope_row['human_key'])}, saved_at={str(current_scope_row['saved_at'])})"  # Build one readable current-scope saved-label summary.

            fig, axis = plt.subplots(figsize=(4.5, 4.5))  # Create one square figure for the current image.
            if image_array.ndim == 2:  # Check whether the current image is grayscale.
                axis.imshow(image_array, cmap="gray")  # Display the grayscale image with a grayscale colormap.
            else:  # Otherwise display the image as a regular color image.
                axis.imshow(image_array)  # Display the color image directly.
            axis.set_title(f"stim_ID {stim_id}  |  test_pos {test_pos}")  # Add the stim_ID and CNN test position above the image.
            axis.axis("off")  # Hide the axes around the image.
            plt.show()  # Display the image figure inline.

            print(f"Correct label: {true_label}")  # Show the correct label under the image.
            print(f"Selected-group predictions: {str(item_row['shared_pred_label_signature'])}")  # Show the combined selected-group predictions under the image.
            if previous_human_label != "":  # Show the newest reused earlier human label when one exists.
                print(f"Previous reused human label: {previous_human_label}  |  saved_at: {previous_human_saved_at}")  # Print the reused earlier human-label summary.
            else:  # Otherwise say that no reused earlier human label exists for this image.
                print("Previous reused human label: none")  # Make the absence of a reused earlier label explicit.
            if current_scope_label_text != "":  # Show the saved label from this current scope when one exists.
                print(f"Current scope saved label: {current_scope_label_text}")  # Print the current-scope saved-label summary.
            else:  # Otherwise say that this current scope does not yet have a saved label for this image.
                print("Current scope saved label: none")  # Make the absence of a current-scope label explicit.

            print()  # Add a blank line before the selected-group prediction table.
            print("Selected-group prediction table")  # Label the prediction-table section.
            display(predictions_df)  # Show the selected-group prediction table inline.

    def move_prev(_button):  # Move to the previous visible review item when possible.
        if int(review_index_slider.value) > int(review_index_slider.min):  # Move backward only if an earlier visible item exists.
            review_index_slider.value = int(review_index_slider.value) - 1  # Decrease the slider value by one step.

    def move_next(_button):  # Move to the next visible review item when possible.
        if int(review_index_slider.value) < int(review_index_slider.max):  # Move forward only if a later visible item exists.
            review_index_slider.value = int(review_index_slider.value) + 1  # Increase the slider value by one step.

    def handle_label_entry_change(change):  # Save the typed key when the text-box value changes and Enter confirms the submission.
        if change["name"] != "value":  # Ignore changes that are not value updates.
            return  # Stop this callback here.
        typed_value = str(change["new"]).strip().lower()  # Read the newly submitted key from the text box.
        if typed_value == "":  # Ignore empty submissions.
            return  # Stop this callback here.
        save_current_label_from_key(typed_value)  # Save the submitted key as the current human label immediately.
        label_entry.value = ""  # Clear the text box after saving so the next key can be entered cleanly.

    def handle_review_mode_change(change):  # Refresh the visible review sequence when the review mode changes.
        if change["name"] == "value":  # React only to review-mode value changes.
            refresh_review_slider()  # Rebuild the slider range for the new review mode.
            render_current_review_item()  # Redraw the current visible review item.

    review_mode_dropdown = widgets.Dropdown(  # Create the dropdown that chooses whether to view only pending items or all scope items.
        options=[("Pending only", "pending"), ("All scope items", "all")],  # Offer the two review modes used in your earlier notebook.
        value="pending",  # Start in pending-only mode so only still-unlabeled shared-scope images appear first.
        description="Mode:",  # Add a short label to the left of the dropdown.
        layout=widgets.Layout(width="260px"),  # Give the dropdown a readable width.
    )  # Finish the review-mode dropdown.

    review_index_slider = widgets.IntSlider(  # Create the slider that chooses the visible item within the current review sequence.
        value=0,  # Start at the first visible item.
        min=0,  # Start the slider minimum at zero.
        max=0,  # Start the slider maximum at zero until the visible sequence is built.
        step=1,  # Move one visible review item at a time.
        description="Item:",  # Add a short label to the left of the slider.
        continuous_update=False,  # Redraw only when the slider selection is committed.
        layout=widgets.Layout(width="500px"),  # Give the slider a readable width.
    )  # Finish the review-index slider.

    prev_button = widgets.Button(description="Previous")  # Create the button that moves to the previous visible review item.
    next_button = widgets.Button(description="Next")  # Create the button that moves to the next visible review item.

    label_entry = widgets.Text(  # Create the text box where the user types one review key and presses Enter to save immediately.
        value="",  # Start with an empty text box.
        description="Label key:",  # Add a short label to the left of the text box.
        placeholder="Type one key then press Enter",  # Explain how to use the text box.
        layout=widgets.Layout(width="340px"),  # Give the text box a readable width.
    )  # Finish the label-entry text box.
    label_entry.continuous_update = False  # Treat Enter as the meaningful submission event for the text-box value change.

    review_output = widgets.Output()  # Create the output area for the current image and its label information.
    status_output = widgets.Output()  # Create the output area for save confirmations and small warnings.

    review_index_slider.observe(render_current_review_item, names="value")  # Redraw the current review item whenever the slider value changes.
    review_mode_dropdown.observe(handle_review_mode_change, names="value")  # Refresh the visible review sequence whenever the review mode changes.
    prev_button.on_click(move_prev)  # Connect the Previous button to its callback.
    next_button.on_click(move_next)  # Connect the Next button to its callback.
    label_entry.observe(handle_label_entry_change, names="value")  # Save the typed key when Enter confirms a new text-box value.

    refresh_review_slider()  # Build the first visible-sequence slider range.
    render_current_review_item()  # Draw the first visible review item immediately.

    display(HTML("<b>Human-review viewer.</b> Type one key into <i>Label key</i>, then press Enter to save immediately. The new label is saved right away into this scope and into this notebook's reusable cache."))  # Explain how the viewer works.
    display(widgets.HBox([review_mode_dropdown, review_index_slider]))  # Show the review-mode dropdown and the item slider together.
    display(widgets.HBox([prev_button, next_button, label_entry]))  # Show the Previous button, Next button, and label-entry text box together.
    display(status_output)  # Show the status-output area under the controls.
    display(review_output)  # Show the main review-output area under the status messages.


In [ ]:
# Cell 11 — Finalize shared human labels and build the hybrid test set  # 03.26.2026 Add mode-aware skipping for automated-label and reload-only runs.
# Combine reused and newly created human labels for the shared misclassification set and build the hybrid evaluation tables only when human-review mode is enabled and fresh analysis is active.  # 03.26.2026 Explain the updated purpose of this cell.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

if bool(USER_CONFIG["load_saved_results_only"]):  # 03.26.2026 Skip hybrid-table construction when the notebook is set to reload saved outputs only.
    FINAL_SHARED_HUMAN_LABELS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder finalized human-label table in reload-only mode.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared-scope final-label table in reload-only mode.
    HYBRID_TEST_WIDE_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder hybrid one-row-per-stimulus table in reload-only mode.
    HYBRID_PREDICTIONS_LONG_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder hybrid long prediction table in reload-only mode.
    print("Load-saved-results-only is enabled, so Cell 11 skipped hybrid-label construction.")  # 03.26.2026 Confirm that hybrid-label construction was skipped intentionally.
elif not bool(USER_CONFIG["use_human_review"]):  # 03.26.2026 Skip hybrid-table construction entirely when automated-label mode is selected.
    FINAL_SHARED_HUMAN_LABELS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder finalized human-label table in automated-label mode.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder shared-scope final-label table in automated-label mode.
    HYBRID_TEST_WIDE_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder hybrid one-row-per-stimulus table in automated-label mode.
    HYBRID_PREDICTIONS_LONG_DF = pd.DataFrame()  # 03.26.2026 Use an empty placeholder hybrid long prediction table in automated-label mode.
    print("Human review is disabled, so Cell 11 skipped hybrid-label construction.")  # 03.26.2026 Confirm that hybrid-label construction was skipped intentionally.
else:  # 03.26.2026 Run the existing hybrid-label construction path when human-review mode is enabled.
    # Cell 11 — Finalize shared human labels and build the hybrid test set
    # Combine reused and newly created human labels for the shared misclassification set, require full coverage of that set, and build the hybrid evaluation tables.

    if "build_combined_scope_labels_df" not in globals():  # Make sure the revised interactive review cell was run before building the hybrid set.
        raise RuntimeError("Please run Cell 10 before running this cell.")  # Stop with a clear message if the current-scope label functions are missing.

    FINAL_SHARED_HUMAN_LABELS_DF = build_combined_scope_labels_df().copy()  # Load the current scope's combined human labels, including reused and newly reviewed labels.
    FINAL_SHARED_HUMAN_LABELS_DF = FINAL_SHARED_HUMAN_LABELS_DF.sort_values(["test_pos", "stim_ID"], kind="mergesort").reset_index(drop=True)  # Put the combined scope-label table into stable CNN test order.

    required_final_label_columns = ["stim_ID", "stimulus_index", "test_pos", "true_label", "human_label", "human_label_int", "human_key", "saved_at", "source_type", "source_name"]  # List the combined-label columns that must exist for the hybrid-set construction.
    for column_name in required_final_label_columns:  # Check each required combined-label column one at a time.
        if column_name not in FINAL_SHARED_HUMAN_LABELS_DF.columns:  # Stop if any required combined-label column is missing.
            raise KeyError(f"Required final shared human-label column not found: {column_name}")  # Give a clear error that names the missing column.

    shared_scope_stim_ids = set(SHARED_MISCLASS_DF["stim_ID"].astype(int).tolist())  # Build the set of stimulus IDs that belong to the shared misclassification scope.
    final_label_stim_ids = set(FINAL_SHARED_HUMAN_LABELS_DF["stim_ID"].astype(int).tolist()) if len(FINAL_SHARED_HUMAN_LABELS_DF) > 0 else set()  # Build the set of stimulus IDs that currently have a saved human label in this scope.

    missing_shared_label_ids = sorted(list(shared_scope_stim_ids - final_label_stim_ids))  # Find any shared-scope images that still do not have a current-scope human label.
    extra_final_label_ids = sorted(list(final_label_stim_ids - shared_scope_stim_ids))  # Find any saved scope labels that do not belong to the current shared scope.

    if len(missing_shared_label_ids) > 0:  # Stop if any shared misclassification images still lack a current-scope human label.
        raise RuntimeError(f"The hybrid set cannot be built yet because some shared-scope images still have no human label. Missing stim_ID values: {missing_shared_label_ids[:20]}")  # Explain exactly why the hybrid set cannot be created yet.

    if len(extra_final_label_ids) > 0:  # Stop if the current scope label table contains unexpected stimulus IDs.
        raise RuntimeError(f"The current scope label table contains unexpected stim_ID values outside the shared scope: {extra_final_label_ids[:20]}")  # Explain the scope-mismatch problem clearly.

    SHARED_SCOPE_WITH_FINAL_LABELS_DF = SHARED_MISCLASS_DF.copy()  # Copy the shared misclassification table so the final human labels can be merged into it.
    merge_columns = ["stim_ID", "stimulus_index", "test_pos", "human_label", "human_label_int", "human_key", "saved_at", "source_type", "source_name"]  # Keep only the human-label columns needed for the final shared-scope merge.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF = SHARED_SCOPE_WITH_FINAL_LABELS_DF.merge(FINAL_SHARED_HUMAN_LABELS_DF[merge_columns], on=["stim_ID", "stimulus_index", "test_pos"], how="left", validate="one_to_one")  # Merge the final human labels into the shared misclassification review scope.

    if SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label"].isna().any():  # Double-check that every shared-scope row really received a final human label.
        bad_rows = SHARED_SCOPE_WITH_FINAL_LABELS_DF.loc[SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label"].isna(), ["stim_ID", "stimulus_index", "test_pos"]]  # Collect the still-unlabeled shared-scope rows for debugging.
        raise RuntimeError(f"Some shared-scope rows still do not have a human label after the merge:\n{bad_rows.head(20)}")  # Stop with a clear error if any shared-scope row is still unlabeled.

    SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label"] = SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label"].astype(str).str.strip()  # Clean the final human-label text values.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label_int"] = pd.to_numeric(SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label_int"], errors="raise").astype(int)  # Convert the final human-label integers into true integer type.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF["original_true_label"] = SHARED_SCOPE_WITH_FINAL_LABELS_DF["true_label"].astype(str).str.strip()  # Preserve the original metadata true label under a clearer name.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF["original_true_label_int"] = SHARED_SCOPE_WITH_FINAL_LABELS_DF["true_label_int"].astype(int)  # Preserve the original metadata integer true label under a clearer name.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF["hybrid_true_label"] = SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label"]  # Use the human-reviewed label as the hybrid true label inside the shared scope.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF["hybrid_true_label_int"] = SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label_int"]  # Use the human-reviewed integer label as the hybrid true label inside the shared scope.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label_differs_from_original"] = SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label"] != SHARED_SCOPE_WITH_FINAL_LABELS_DF["original_true_label"]  # Mark whether the human-reviewed label differs from the original metadata label.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF["hybrid_label_source"] = "human_review_shared_scope"  # Mark the hybrid-label source for all shared-scope rows.

    HYBRID_TEST_WIDE_DF = COMPARISON_WIDE_DF.copy()  # Copy the one-row-per-stimulus comparison table so hybrid labels can be added without changing the original table.
    HYBRID_TEST_WIDE_DF["is_in_shared_scope"] = HYBRID_TEST_WIDE_DF["stim_ID"].astype(int).isin(shared_scope_stim_ids)  # Mark which full-test rows belong to the shared misclassification scope.

    shared_merge_columns = ["stim_ID", "human_label", "human_label_int", "human_key", "saved_at", "source_type", "source_name", "human_label_differs_from_original"]  # Keep only the shared-scope human-label columns needed in the full hybrid table.
    HYBRID_TEST_WIDE_DF = HYBRID_TEST_WIDE_DF.merge(SHARED_SCOPE_WITH_FINAL_LABELS_DF[shared_merge_columns], on="stim_ID", how="left", validate="one_to_one")  # Merge the final shared-scope human labels into the full one-row-per-stimulus table.

    HYBRID_TEST_WIDE_DF["hybrid_true_label"] = np.where(HYBRID_TEST_WIDE_DF["is_in_shared_scope"], HYBRID_TEST_WIDE_DF["human_label"], HYBRID_TEST_WIDE_DF["true_label"])  # Use the human-reviewed label inside the shared scope and the original metadata label outside it.
    HYBRID_TEST_WIDE_DF["hybrid_true_label_int"] = np.where(HYBRID_TEST_WIDE_DF["is_in_shared_scope"], HYBRID_TEST_WIDE_DF["human_label_int"], HYBRID_TEST_WIDE_DF["true_label_int"])  # Build the matching integer hybrid true-label column.
    HYBRID_TEST_WIDE_DF["hybrid_label_source"] = np.where(HYBRID_TEST_WIDE_DF["is_in_shared_scope"], "human_review_shared_scope", "original_metadata_label")  # Record where each hybrid true label came from.
    HYBRID_TEST_WIDE_DF["hybrid_true_label"] = HYBRID_TEST_WIDE_DF["hybrid_true_label"].astype(str).str.strip()  # Clean the final hybrid true-label text values.
    HYBRID_TEST_WIDE_DF["hybrid_true_label_int"] = pd.to_numeric(HYBRID_TEST_WIDE_DF["hybrid_true_label_int"], errors="raise").astype(int)  # Convert the final hybrid integer labels into true integer type.

    if HYBRID_TEST_WIDE_DF.loc[HYBRID_TEST_WIDE_DF["is_in_shared_scope"], "hybrid_true_label"].isin(["", "nan", "None"]).any():  # Double-check that no shared-scope rows kept a missing hybrid label.
        raise RuntimeError("At least one shared-scope image still has a missing hybrid true label after the full-test merge.")  # Stop if the shared scope is not fully labeled inside the hybrid test table.

    HYBRID_TEST_WIDE_DF["hybrid_label_differs_from_original"] = HYBRID_TEST_WIDE_DF["hybrid_true_label"] != HYBRID_TEST_WIDE_DF["true_label"]  # Mark which full-test rows changed label under the hybrid evaluation scheme.

    hybrid_merge_columns = ["stimulus_index", "stim_ID", "hybrid_true_label", "hybrid_true_label_int", "hybrid_label_source", "hybrid_label_differs_from_original", "is_in_shared_scope"]  # Keep only the hybrid-label columns needed in the long per-prediction table.
    HYBRID_PREDICTIONS_LONG_DF = ALL_PREDICTIONS_LONG_DF.copy()  # Copy the long per-group prediction table so hybrid correctness can be computed without changing the original table.
    HYBRID_PREDICTIONS_LONG_DF = HYBRID_PREDICTIONS_LONG_DF.merge(HYBRID_TEST_WIDE_DF[hybrid_merge_columns], on=["stimulus_index", "stim_ID"], how="left", validate="many_to_one")  # Merge the hybrid true-label columns into the long per-group prediction table.

    HYBRID_PREDICTIONS_LONG_DF["original_is_correct"] = HYBRID_PREDICTIONS_LONG_DF["is_correct"].astype(bool)  # Preserve the original correctness flag under a clearer name.
    HYBRID_PREDICTIONS_LONG_DF["hybrid_is_correct"] = HYBRID_PREDICTIONS_LONG_DF["pred_label"].astype(str).str.strip() == HYBRID_PREDICTIONS_LONG_DF["hybrid_true_label"].astype(str).str.strip()  # Recompute correctness using the hybrid true label.
    HYBRID_PREDICTIONS_LONG_DF["became_correct_under_hybrid"] = (~HYBRID_PREDICTIONS_LONG_DF["original_is_correct"]) & HYBRID_PREDICTIONS_LONG_DF["hybrid_is_correct"]  # Mark predictions that became correct under the hybrid labeling scheme.
    HYBRID_PREDICTIONS_LONG_DF["became_incorrect_under_hybrid"] = HYBRID_PREDICTIONS_LONG_DF["original_is_correct"] & (~HYBRID_PREDICTIONS_LONG_DF["hybrid_is_correct"])  # Mark predictions that became incorrect under the hybrid labeling scheme.

    HYBRID_SCOPE_SUMMARY_DF = pd.DataFrame([  # Build one short summary row that describes the finalized shared-scope human labels and the hybrid test set.
        {
            "review_scope_name": REVIEW_SCOPE_NAME,  # Record the current review-scope name.
            "n_full_test_rows": len(HYBRID_TEST_WIDE_DF),  # Record the total number of stimuli in the CNN test set.
            "n_shared_scope_rows": len(SHARED_SCOPE_WITH_FINAL_LABELS_DF),  # Record the size of the shared misclassification scope.
            "n_final_shared_human_labels": len(FINAL_SHARED_HUMAN_LABELS_DF),  # Record the number of final human labels in the shared scope.
            "n_shared_labels_changed_from_original": int(SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label_differs_from_original"].sum()),  # Record how many shared-scope labels changed relative to the original metadata label.
            "n_shared_labels_same_as_original": int((~SHARED_SCOPE_WITH_FINAL_LABELS_DF["human_label_differs_from_original"]).sum()),  # Record how many shared-scope labels stayed the same as the original metadata label.
        }
    ])  # Finish the hybrid-scope summary table.

    HYBRID_SCOPE_LABEL_CHANGE_COUNTS_DF = SHARED_SCOPE_WITH_FINAL_LABELS_DF.groupby(["original_true_label", "human_label"], dropna=False).size().reset_index(name="n_images")  # Count how often each original label became each human-reviewed label inside the shared scope.
    HYBRID_SCOPE_LABEL_CHANGE_COUNTS_DF = HYBRID_SCOPE_LABEL_CHANGE_COUNTS_DF.sort_values(["n_images", "original_true_label", "human_label"], ascending=[False, True, True]).reset_index(drop=True)  # Put the shared-scope label-change counts into a stable readable order.

    NOTEBOOK_TABLES_DIR = NOTEBOOK_OUTPUT_DIR / "tables"  # Create one folder for saved output tables from this notebook.
    NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)  # Create the tables folder if it does not already exist.
    CURRENT_SCOPE_TABLES_DIR = CURRENT_SCOPE_DIR / "tables"  # Create one tables folder for outputs specific to this current shared review scope.
    CURRENT_SCOPE_TABLES_DIR.mkdir(parents=True, exist_ok=True)  # Create the current-scope tables folder if it does not already exist.

    FINAL_SHARED_HUMAN_LABELS_EXPORT_PATH = CURRENT_SCOPE_TABLES_DIR / "final_shared_human_labels.csv"  # Save the finalized shared-scope human labels to this CSV file.
    SHARED_SCOPE_WITH_FINAL_LABELS_PATH = CURRENT_SCOPE_TABLES_DIR / "shared_scope_with_final_human_labels.csv"  # Save the merged shared-scope table with original and human labels here.
    HYBRID_TEST_WIDE_PATH = NOTEBOOK_TABLES_DIR / "hybrid_test_wide.csv"  # Save the full one-row-per-stimulus hybrid test table here.
    HYBRID_PREDICTIONS_LONG_PATH = NOTEBOOK_TABLES_DIR / "hybrid_predictions_long.csv"  # Save the full long per-group hybrid prediction table here.
    HYBRID_SCOPE_SUMMARY_PATH = CURRENT_SCOPE_TABLES_DIR / "hybrid_scope_summary.csv"  # Save the one-row hybrid-scope summary here.
    HYBRID_SCOPE_LABEL_CHANGE_COUNTS_PATH = CURRENT_SCOPE_TABLES_DIR / "hybrid_scope_label_change_counts.csv"  # Save the shared-scope label-change count table here.

    final_shared_export_columns = [column_name for column_name in ["review_order", "scope_name", "stim_ID", "stimulus_index", "test_pos", "true_label", "human_key", "human_label", "human_label_int", "saved_at", "selected_run_keys", "previous_cache_hit", "source_type", "source_name"] if column_name in FINAL_SHARED_HUMAN_LABELS_DF.columns]  # Keep a clean set of export columns for the final shared human-label file.
    FINAL_SHARED_HUMAN_LABELS_DF[final_shared_export_columns].to_csv(FINAL_SHARED_HUMAN_LABELS_EXPORT_PATH, index=False)  # Save the clean finalized shared human-label table.
    SHARED_SCOPE_WITH_FINAL_LABELS_DF.to_csv(SHARED_SCOPE_WITH_FINAL_LABELS_PATH, index=False)  # Save the merged shared-scope original-vs-human label table.
    HYBRID_TEST_WIDE_DF.to_csv(HYBRID_TEST_WIDE_PATH, index=False)  # Save the full one-row-per-stimulus hybrid test table.
    HYBRID_PREDICTIONS_LONG_DF.to_csv(HYBRID_PREDICTIONS_LONG_PATH, index=False)  # Save the full long per-group hybrid prediction table.
    HYBRID_SCOPE_SUMMARY_DF.to_csv(HYBRID_SCOPE_SUMMARY_PATH, index=False)  # Save the one-row hybrid-scope summary table.
    HYBRID_SCOPE_LABEL_CHANGE_COUNTS_DF.to_csv(HYBRID_SCOPE_LABEL_CHANGE_COUNTS_PATH, index=False)  # Save the shared-scope label-change count table.

    print("The hybrid test set was built successfully.")  # Confirm that the finalized human labels and hybrid evaluation tables were created.
    print("Final shared human-label file:", FINAL_SHARED_HUMAN_LABELS_EXPORT_PATH)  # Show where the finalized shared human-label table was saved.
    print("Hybrid test table:", HYBRID_TEST_WIDE_PATH)  # Show where the one-row-per-stimulus hybrid test table was saved.
    print("Hybrid long prediction table:", HYBRID_PREDICTIONS_LONG_PATH)  # Show where the long per-group hybrid prediction table was saved.
    display(HYBRID_SCOPE_SUMMARY_DF)  # Show the one-row summary of the finalized shared human labels and hybrid test set.
    display(HYBRID_SCOPE_LABEL_CHANGE_COUNTS_DF.head(20))  # Show the most common original-to-human label changes inside the shared scope.
    display(SHARED_SCOPE_WITH_FINAL_LABELS_DF[["review_order", "test_pos", "stim_ID", "original_true_label", "human_label", "human_label_differs_from_original", "source_type", "source_name"]].head(20))  # Show the first few finalized shared-scope human-label rows.


In [ ]:
# Cell 12 — Build, display, and save confusion matrices for the active evaluation mode  # 03.26.2026 Generalize this cell for auto-only versus human-review labels, merged versus separate gray classes, and reload-only figure rebuilding.
# Use the saved user choices to compute or reload the active confusion-matrix outputs, then rebuild the figures with the requested cividis colormap.  # 03.26.2026 Explain the purpose of this configurable evaluation cell.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

if "cividis" not in plt.colormaps():  # 03.26.2026 Stop if the requested confusion-matrix colormap is unavailable in this Python environment.
    raise ValueError("The 'cividis' colormap is not available in this Python environment, so Cell 12 cannot use the requested confusion-matrix style.")  # 03.26.2026 Explain the exact style limitation clearly.

SELECTED_GROUPS = list(USER_CONFIG["groups"])  # 03.26.2026 Keep the selected CNN groups available as a direct global variable in this cell.
EVAL_CHOICE = str(USER_CONFIG["eval_choice"])  # 03.26.2026 Keep the chosen CNN evaluation type available as a direct global variable in this cell.
NOTEBOOK_OUTPUT_DIR = Path(USER_CONFIG["output_dir"])  # 03.26.2026 Keep the selected notebook output directory available as a direct global variable in this cell.
EVAL_CLASS_ORDER = list(USER_CONFIG["eval_class_order"])  # 03.26.2026 Read the active evaluation class order from the saved user choices.
EVAL_MODE_SUFFIX = str(USER_CONFIG["eval_mode_suffix"])  # 03.26.2026 Read the short gray-handling mode label from the saved user choices.
LABEL_SOURCE_SUFFIX = str(USER_CONFIG["label_source_suffix"])  # 03.26.2026 Read the short label-source mode label from the saved user choices.
NOTEBOOK_SUMMARY_FILE_NAME = str(USER_CONFIG["notebook_summary_file_name"])  # 03.26.2026 Read the active notebook-level long-summary filename from the saved user choices.
EVAL_OUTPUT_FOLDER_NAME = str(USER_CONFIG["eval_output_folder_name"])  # 03.26.2026 Read the active figure-output folder name from the saved user choices.
COMBINE_GRAY_CLASSES = bool(USER_CONFIG["combine_gray_classes"])  # 03.26.2026 Read whether gray_d and gray_l should be merged into one gray evaluation class.
USE_HUMAN_REVIEW = bool(USER_CONFIG["use_human_review"])  # 03.26.2026 Read whether this evaluation should use human-reviewed labels or automated labels only.
LOAD_SAVED_RESULTS_ONLY = bool(USER_CONFIG["load_saved_results_only"])  # 03.26.2026 Read whether this cell should reload saved outputs instead of recomputing them.

summaries_dir = NOTEBOOK_OUTPUT_DIR / "summaries"  # 03.26.2026 Build the notebook-level summaries folder path.
summaries_dir.mkdir(parents=True, exist_ok=True)  # 03.26.2026 Make sure the summaries folder exists before any files are saved.
eval_output_dir = NOTEBOOK_OUTPUT_DIR / EVAL_OUTPUT_FOLDER_NAME  # 03.26.2026 Build the active evaluation-output folder path.
eval_output_dir.mkdir(parents=True, exist_ok=True)  # 03.26.2026 Make sure the active evaluation-output folder exists before any files are saved.
summary_long_path = summaries_dir / NOTEBOOK_SUMMARY_FILE_NAME  # 03.26.2026 Build the saved long-summary path for the active evaluation mode.
accuracy_summary_path = eval_output_dir / f"{LABEL_SOURCE_SUFFIX}_{EVAL_MODE_SUFFIX}_accuracy_summary.csv"  # 03.26.2026 Build the short accuracy-summary path for the active evaluation mode.

def evaluation_label_from_text(label_text):  # 03.26.2026 Convert one source label into the active evaluation label after applying the gray-handling choice.
    label_text = str(label_text).strip()  # 03.26.2026 Normalize the incoming label text before evaluation mapping is applied.
    if COMBINE_GRAY_CLASSES and label_text in {"gray_d", "gray_l"}:  # 03.26.2026 Collapse both gray subclasses into one gray label when gray merging is enabled.
        return "gray"  # 03.26.2026 Return the merged gray label for either gray subclass.
    return label_text  # 03.26.2026 Leave every non-gray label unchanged for evaluation.

def build_eval_cm_long_df(y_true_labels, y_pred_labels, group_name, accuracy_value, review_scope_name, n_human_reviewed_images):  # 03.26.2026 Build the long-format confusion-matrix table for one CNN group in the active evaluation mode.
    cm_counts = confusion_matrix(y_true_labels, y_pred_labels, labels=EVAL_CLASS_ORDER)  # 03.26.2026 Build the raw confusion-matrix counts using the active evaluation class order.
    row_totals = cm_counts.sum(axis=1).astype(np.int64)  # 03.26.2026 Count how many items belong to each true-label row in the active evaluation.
    rows = []  # 03.26.2026 Collect one long-format row per confusion-matrix cell.
    for true_idx, true_label in enumerate(EVAL_CLASS_ORDER):  # 03.26.2026 Walk through each true-label row in the active evaluation order.
        for pred_idx, pred_label in enumerate(EVAL_CLASS_ORDER):  # 03.26.2026 Walk through each predicted-label column in the active evaluation order.
            raw_count = int(cm_counts[true_idx, pred_idx])  # 03.26.2026 Read the raw count in this confusion-matrix cell.
            true_class_total = int(row_totals[true_idx])  # 03.26.2026 Read the total number of images in this true-label row.
            percent_value = (float(raw_count) / float(true_class_total) * 100.0) if true_class_total > 0 else 0.0  # 03.26.2026 Convert the raw count into a row-normalized percentage.
            rows.append({"model": "CNN", "group": str(group_name), "true_label": str(true_label), "pred_label": str(pred_label), "is_error": bool(true_label != pred_label), "raw_count": raw_count, "true_class_total": true_class_total, "percent_of_true_class": percent_value, "final_test_accuracy": float(accuracy_value), "summary_type": str(LABEL_SOURCE_SUFFIX), "review_scope_name": str(review_scope_name), "n_human_reviewed_images": int(n_human_reviewed_images), "n_test_images_total": int(len(y_true_labels))})  # 03.26.2026 Save one long-format confusion-matrix row with the existing summary-style columns.
    return pd.DataFrame(rows)  # 03.26.2026 Return the completed long-format confusion-matrix table.

def build_percent_matrix_from_long(long_df, group_name):  # 03.26.2026 Convert saved long-format rows back into one row-normalized percentage matrix for one CNN group.
    sub_df = long_df.loc[(long_df["model"].astype(str) == "CNN") & (long_df["group"].astype(str) == str(group_name)), ["true_label", "pred_label", "percent_of_true_class"]].copy()  # 03.26.2026 Keep only the rows for this one CNN group.
    if len(sub_df) == 0:  # 03.26.2026 Stop if the requested CNN group rows are missing from the long-format summary.
        raise RuntimeError("No long-format confusion-matrix rows were found for CNN / " + str(group_name))  # 03.26.2026 Explain which group is missing from the long summary.
    matrix_df = sub_df.pivot(index="true_label", columns="pred_label", values="percent_of_true_class").reindex(index=EVAL_CLASS_ORDER, columns=EVAL_CLASS_ORDER).fillna(0.0)  # 03.26.2026 Rebuild the percentage matrix in the active evaluation order.
    return matrix_df.to_numpy(dtype=np.float64)  # 03.26.2026 Return the rebuilt percentage matrix as a NumPy array.

def save_confusion_matrix_png(cm_pct, out_path, title_text, annotate_numbers):  # 03.26.2026 Save one row-normalized confusion matrix as a PNG using the requested cividis colormap.
    fig, ax = plt.subplots(figsize=(7, 7))  # 03.26.2026 Create one square figure for the confusion matrix.
    image = ax.imshow(cm_pct, cmap="cividis", vmin=0.0, vmax=100.0)  # 03.26.2026 Draw the row-normalized percentages using the requested cividis colormap.
    ax.set_xticks(np.arange(len(EVAL_CLASS_ORDER)))  # 03.26.2026 Put one tick at each predicted-label column in the active evaluation order.
    ax.set_yticks(np.arange(len(EVAL_CLASS_ORDER)))  # 03.26.2026 Put one tick at each true-label row in the active evaluation order.
    ax.set_xticklabels(EVAL_CLASS_ORDER, rotation=45, ha="right", rotation_mode="anchor")  # 03.26.2026 Show the predicted labels on the x axis.
    ax.set_yticklabels(EVAL_CLASS_ORDER)  # 03.26.2026 Show the true labels on the y axis.
    ax.set_xlabel("Predicted label")  # 03.26.2026 Label the x axis.
    ax.set_ylabel("True label")  # 03.26.2026 Label the y axis.
    ax.set_title(title_text)  # 03.26.2026 Add the descriptive figure title.
    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)  # 03.26.2026 Add the color scale bar.
    colorbar.set_label("Percent of true class")  # 03.26.2026 Label the colorbar.
    if annotate_numbers:  # 03.26.2026 Write the numeric percentage into each cell when requested.
        for row_idx in range(cm_pct.shape[0]):  # 03.26.2026 Loop over the true-label rows.
            for col_idx in range(cm_pct.shape[1]):  # 03.26.2026 Loop over the predicted-label columns.
                value = float(cm_pct[row_idx, col_idx])  # 03.26.2026 Read the current cell percentage.
                text_color = "white" if value >= 50.0 else "black"  # 03.26.2026 Choose a readable text color for the current background shade.
                ax.text(col_idx, row_idx, f"{value:.1f}", ha="center", va="center", color=text_color, fontsize=10)  # 03.26.2026 Write the rounded percentage value into the cell.
    fig.subplots_adjust(left=0.22, bottom=0.20, right=0.92, top=0.90)  # 03.26.2026 Reserve room so labels and the colorbar are not clipped.
    fig.savefig(out_path, dpi=150)  # 03.26.2026 Save the confusion-matrix figure to disk.
    plt.close(fig)  # 03.26.2026 Close the figure so notebook memory does not keep growing.

if LOAD_SAVED_RESULTS_ONLY:  # 03.26.2026 Reload the previously saved notebook-level summary instead of recomputing it.
    if not summary_long_path.exists():  # 03.26.2026 Stop if the saved notebook-level long summary for the active mode is missing.
        raise FileNotFoundError("The saved long-summary file for the active mode was not found: " + str(summary_long_path))  # 03.26.2026 Explain which saved file is required for reload-only mode.
    summary_long_df = pd.read_csv(summary_long_path)  # 03.26.2026 Load the previously saved long-format confusion-matrix summary table for the active mode.
else:  # 03.26.2026 Compute the active long-format confusion-matrix summary table during a fresh notebook run.
    if USE_HUMAN_REVIEW:  # 03.26.2026 Build the active confusion matrices from the hybrid human-review tables.
        if "HYBRID_PREDICTIONS_LONG_DF" not in globals() or len(HYBRID_PREDICTIONS_LONG_DF) == 0:  # 03.26.2026 Require the hybrid long prediction table before building human-review confusion matrices.
            raise RuntimeError("HYBRID_PREDICTIONS_LONG_DF is missing. Please complete the human-review cells before running Cell 12 in human-review mode.")  # 03.26.2026 Explain which earlier cells are required for human-review mode.
        source_df = HYBRID_PREDICTIONS_LONG_DF.copy()  # 03.26.2026 Use the hybrid long prediction table as the source for the active evaluation.
        true_label_column = "hybrid_true_label"  # 03.26.2026 Use the hybrid true labels when human review mode is enabled.
        review_scope_name = str(REVIEW_SCOPE_NAME) if "REVIEW_SCOPE_NAME" in globals() else ""  # 03.26.2026 Carry the active review-scope name into the saved long summary.
        n_human_reviewed_images = int(len(FINAL_SHARED_HUMAN_LABELS_DF)) if "FINAL_SHARED_HUMAN_LABELS_DF" in globals() else 0  # 03.26.2026 Count how many finalized human-review labels were used in this evaluation.
    else:  # 03.26.2026 Build the active confusion matrices from the original automated-label tables.
        if "ALL_PREDICTIONS_LONG_DF" not in globals() or len(ALL_PREDICTIONS_LONG_DF) == 0:  # 03.26.2026 Require the original long prediction table before building automated-label confusion matrices.
            raise RuntimeError("ALL_PREDICTIONS_LONG_DF is missing. Please rerun the earlier loading cells before running Cell 12 in automated-label mode.")  # 03.26.2026 Explain which earlier cells are required for automated-label mode.
        source_df = ALL_PREDICTIONS_LONG_DF.copy()  # 03.26.2026 Use the original long prediction table as the source for the active evaluation.
        true_label_column = "true_label"  # 03.26.2026 Use the original metadata true labels when automated-label mode is enabled.
        review_scope_name = ""  # 03.26.2026 Keep the review-scope name blank in automated-label mode.
        n_human_reviewed_images = 0  # 03.26.2026 Record that no human-review labels were used in automated-label mode.
    summary_frames = []  # 03.26.2026 Collect one long-format confusion-matrix table per selected CNN group.
    for group_name in SELECTED_GROUPS:  # 03.26.2026 Build the active confusion-matrix summary for each selected CNN group.
        group_df = source_df.loc[source_df["group"].astype(str) == str(group_name)].copy()  # 03.26.2026 Keep only the rows for this selected CNN group.
        if len(group_df) == 0:  # 03.26.2026 Stop if the active source table has no rows for a selected group.
            raise RuntimeError("No active evaluation rows were found for group: " + str(group_name))  # 03.26.2026 Explain which selected group is missing.
        y_true_labels = [evaluation_label_from_text(value) for value in group_df[true_label_column].tolist()]  # 03.26.2026 Convert the source true labels into the active evaluation labels.
        y_pred_labels = [evaluation_label_from_text(value) for value in group_df["pred_label"].tolist()]  # 03.26.2026 Convert the predicted labels into the active evaluation labels.
        accuracy_value = float(np.mean(np.asarray(y_true_labels, dtype=object) == np.asarray(y_pred_labels, dtype=object))) if len(y_true_labels) > 0 else 0.0  # 03.26.2026 Compute the final-test accuracy under the active evaluation labels.
        summary_frames.append(build_eval_cm_long_df(y_true_labels, y_pred_labels, group_name, accuracy_value, review_scope_name, n_human_reviewed_images))  # 03.26.2026 Save the long-format confusion-matrix table for this selected group.
    summary_long_df = pd.concat(summary_frames, ignore_index=True)  # 03.26.2026 Stack the selected-group confusion-matrix tables into one long summary table.
    summary_long_df.to_csv(summary_long_path, index=False)  # 03.26.2026 Save the active long-format confusion-matrix summary table to disk.

summary_long_df = summary_long_df.copy()  # 03.26.2026 Work from a dedicated copy of the active long-format confusion-matrix summary table.
summary_long_df["model"] = summary_long_df["model"].astype(str)  # 03.26.2026 Normalize the saved model names as text.
summary_long_df["group"] = summary_long_df["group"].astype(str)  # 03.26.2026 Normalize the saved group names as text.
summary_long_df["true_label"] = summary_long_df["true_label"].astype(str)  # 03.26.2026 Normalize the saved true labels as text.
summary_long_df["pred_label"] = summary_long_df["pred_label"].astype(str)  # 03.26.2026 Normalize the saved predicted labels as text.
summary_long_df["percent_of_true_class"] = pd.to_numeric(summary_long_df["percent_of_true_class"], errors="coerce")  # 03.26.2026 Normalize the saved row-normalized percentages as numeric values.
summary_long_df["final_test_accuracy"] = pd.to_numeric(summary_long_df["final_test_accuracy"], errors="coerce")  # 03.26.2026 Normalize the saved final-test accuracies as numeric values.

available_groups = set(summary_long_df.loc[summary_long_df["model"].astype(str) == "CNN", "group"].astype(str).unique().tolist())  # 03.26.2026 Read the selected CNN groups available in the active long summary.
missing_groups = [str(group_name) for group_name in SELECTED_GROUPS if str(group_name) not in available_groups]  # 03.26.2026 Find any selected groups missing from the active long summary.
if len(missing_groups) > 0:  # 03.26.2026 Stop if any selected group is missing from the active long summary.
    raise RuntimeError("Active long-format confusion-matrix rows are missing for groups: " + ", ".join(missing_groups))  # 03.26.2026 Explain which selected groups are missing.

accuracy_rows = []  # 03.26.2026 Collect one short accuracy-summary row per selected CNN group.
display_records = []  # 03.26.2026 Collect the saved confusion-matrix PNG paths for inline display.
for group_name in SELECTED_GROUPS:  # 03.26.2026 Save the active confusion-matrix figures and accuracy rows for each selected CNN group.
    group_name = str(group_name)  # 03.26.2026 Normalize the selected group name as text.
    cm_pct = build_percent_matrix_from_long(summary_long_df, group_name)  # 03.26.2026 Rebuild the row-normalized percentage matrix for this selected CNN group.
    accuracy_series = summary_long_df.loc[(summary_long_df["model"].astype(str) == "CNN") & (summary_long_df["group"].astype(str) == group_name), "final_test_accuracy"]  # 03.26.2026 Read the saved final-test accuracy for this selected group.
    final_test_accuracy_percent = float(accuracy_series.iloc[0]) * 100.0 if len(accuracy_series) > 0 else np.nan  # 03.26.2026 Convert the saved final-test accuracy into percentage units.
    safe_group_name = "".join([character if character.isalnum() or character in ["-", "_"] else "_" for character in group_name])  # 03.26.2026 Convert the group name into a filename-safe string.
    annotated_png_path = eval_output_dir / f"{LABEL_SOURCE_SUFFIX}_cm_pct_CNN_{safe_group_name}_{EVAL_MODE_SUFFIX}_annotated.png"  # 03.26.2026 Build the annotated confusion-matrix PNG path for this selected group.
    blank_png_path = eval_output_dir / f"{LABEL_SOURCE_SUFFIX}_cm_pct_CNN_{safe_group_name}_{EVAL_MODE_SUFFIX}_blank.png"  # 03.26.2026 Build the blank confusion-matrix PNG path for this selected group.
    title_text = f"{LABEL_SOURCE_SUFFIX} | {EVAL_MODE_SUFFIX} | CNN / {group_name}\nrow-normalized confusion matrix"  # 03.26.2026 Build the descriptive figure title for this selected group.
    save_confusion_matrix_png(cm_pct, annotated_png_path, title_text, True)  # 03.26.2026 Save the annotated confusion-matrix figure to disk.
    save_confusion_matrix_png(cm_pct, blank_png_path, title_text, False)  # 03.26.2026 Save the blank confusion-matrix figure to disk.
    accuracy_rows.append({"model": "CNN", "group": group_name, "final_test_accuracy_percent": final_test_accuracy_percent, "summary_type": LABEL_SOURCE_SUFFIX, "eval_mode": EVAL_MODE_SUFFIX, "annotated_png_path": str(annotated_png_path), "blank_png_path": str(blank_png_path)})  # 03.26.2026 Save one short accuracy-summary row for this selected group.
    display_records.append({"group": group_name, "final_test_accuracy_percent": final_test_accuracy_percent, "annotated_png_path": annotated_png_path, "blank_png_path": blank_png_path})  # 03.26.2026 Save the figure paths needed for inline display.

accuracy_summary_df = pd.DataFrame(accuracy_rows).sort_values(["model", "group"], kind="mergesort").reset_index(drop=True)  # 03.26.2026 Build the one-row-per-group active accuracy-summary table.
accuracy_summary_df.to_csv(accuracy_summary_path, index=False)  # 03.26.2026 Save the active accuracy-summary table to disk.

ACTIVE_EVAL_SUMMARY_LONG_DF = summary_long_df.copy()  # 03.26.2026 Keep the active long-format confusion-matrix summary table available for later cells.
ACTIVE_EVAL_SUMMARY_LONG_PATH = summary_long_path  # 03.26.2026 Keep the saved active long-summary path available for later cells.
ACTIVE_ACCURACY_SUMMARY_DF = accuracy_summary_df.copy()  # 03.26.2026 Keep the active accuracy-summary table available for later cells.
ACTIVE_ACCURACY_SUMMARY_PATH = accuracy_summary_path  # 03.26.2026 Keep the saved active accuracy-summary path available for later cells.
ACTIVE_EVAL_OUTPUT_DIR = eval_output_dir  # 03.26.2026 Keep the active evaluation-output folder available for later cells.

print("Active confusion-matrix outputs saved in:", eval_output_dir.resolve())  # 03.26.2026 Show where the active confusion-matrix figures and tables were saved.
print("Active accuracy summary saved to:", accuracy_summary_path)  # 03.26.2026 Show where the active accuracy-summary CSV was saved.
print("Active long-format confusion summary saved to:", summary_long_path)  # 03.26.2026 Show where the active long-format confusion summary CSV was saved.
display(accuracy_summary_df)  # 03.26.2026 Display the one-row-per-group active accuracy-summary table inline.

for display_record in display_records:  # 03.26.2026 Show each saved confusion matrix inline in the same two-panel style as the existing notebook.
    print()  # 03.26.2026 Add a blank line before each group's figures.
    print(f"Confusion matrices — {LABEL_SOURCE_SUFFIX} — {display_record['group']}")  # 03.26.2026 Label this group's inline confusion-matrix figure section.
    print(f"Final test accuracy: {display_record['final_test_accuracy_percent']:.2f}%")  # 03.26.2026 Show the active final-test accuracy for this selected group.
    annotated_img = plt.imread(str(display_record["annotated_png_path"]))  # 03.26.2026 Read the saved annotated confusion-matrix PNG back from disk.
    blank_img = plt.imread(str(display_record["blank_png_path"]))  # 03.26.2026 Read the saved blank confusion-matrix PNG back from disk.
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # 03.26.2026 Create one figure with two panels for the annotated and blank versions.
    axes[0].imshow(annotated_img)  # 03.26.2026 Draw the annotated confusion-matrix figure.
    axes[0].set_title(f"{display_record['group']} annotated")  # 03.26.2026 Add the annotated-panel title.
    axes[0].axis("off")  # 03.26.2026 Hide the axes around the annotated PNG.
    axes[1].imshow(blank_img)  # 03.26.2026 Draw the blank confusion-matrix figure.
    axes[1].set_title(f"{display_record['group']} blank")  # 03.26.2026 Add the blank-panel title.
    axes[1].axis("off")  # 03.26.2026 Hide the axes around the blank PNG.
    plt.tight_layout()  # 03.26.2026 Adjust the layout so the titles and images fit cleanly.
    plt.show()  # 03.26.2026 Display the two-panel confusion-matrix figure inline.


In [ ]:
# Cell 13 — Build, display, and save pairwise difference matrices for the active evaluation mode  # 03.26.2026 Generalize this cell for auto-only versus human-review labels, merged versus separate gray classes, and reload-only figure rebuilding.
# Create within-model pairwise error-difference matrices from the active confusion-matrix summary, then rebuild the figures with the requested reversed PuOr colormap.  # 03.26.2026 Explain the purpose of this configurable difference-matrix cell.

from matplotlib import colors as mcolors  # 03.26.2026 Import the normalization helper used to center the difference-matrix color scale at zero.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

SELECTED_GROUPS = list(USER_CONFIG["groups"])  # 03.26.2026 Keep the selected CNN groups available as a direct global variable in this cell.
NOTEBOOK_OUTPUT_DIR = Path(USER_CONFIG["output_dir"])  # 03.26.2026 Keep the selected notebook output directory available as a direct global variable in this cell.
EVAL_CLASS_ORDER = list(USER_CONFIG["eval_class_order"])  # 03.26.2026 Read the active evaluation class order from the saved user choices.
EVAL_MODE_SUFFIX = str(USER_CONFIG["eval_mode_suffix"])  # 03.26.2026 Read the short gray-handling mode label from the saved user choices.
LABEL_SOURCE_SUFFIX = str(USER_CONFIG["label_source_suffix"])  # 03.26.2026 Read the short label-source mode label from the saved user choices.
NOTEBOOK_SUMMARY_FILE_NAME = str(USER_CONFIG["notebook_summary_file_name"])  # 03.26.2026 Read the active notebook-level long-summary filename from the saved user choices.
EVAL_OUTPUT_FOLDER_NAME = str(USER_CONFIG["eval_output_folder_name"])  # 03.26.2026 Read the active figure-output folder name from the saved user choices.
eval_output_dir = NOTEBOOK_OUTPUT_DIR / EVAL_OUTPUT_FOLDER_NAME  # 03.26.2026 Build the active evaluation-output folder path.
difference_output_dir = eval_output_dir / "difference_matrices"  # 03.26.2026 Build the active difference-matrix output folder path.
difference_output_dir.mkdir(parents=True, exist_ok=True)  # 03.26.2026 Make sure the active difference-matrix output folder exists before any files are saved.
difference_long_csv_path = difference_output_dir / "difference_matrices_long.csv"  # 03.26.2026 Build the long-format numeric difference-table path for the active mode.
difference_summary_csv_path = difference_output_dir / "difference_matrices_summary.csv"  # 03.26.2026 Build the short difference-summary path for the active mode.

if "ACTIVE_EVAL_SUMMARY_LONG_DF" in globals() and len(ACTIVE_EVAL_SUMMARY_LONG_DF) > 0:  # 03.26.2026 Prefer the in-memory active confusion-matrix summary when Cell 12 already created it.
    summary_long_df = ACTIVE_EVAL_SUMMARY_LONG_DF.copy()  # 03.26.2026 Reuse the in-memory active long-format confusion-matrix summary table.
else:  # 03.26.2026 Reload the active confusion-matrix summary from disk when Cell 12 was not run in this kernel.
    summary_long_path = NOTEBOOK_OUTPUT_DIR / "summaries" / NOTEBOOK_SUMMARY_FILE_NAME  # 03.26.2026 Build the saved active long-summary path from the saved user choices.
    if not summary_long_path.exists():  # 03.26.2026 Stop if the active long-format confusion summary does not exist on disk.
        raise FileNotFoundError("The active long-format confusion summary was not found: " + str(summary_long_path))  # 03.26.2026 Explain which saved summary file is required.
    summary_long_df = pd.read_csv(summary_long_path)  # 03.26.2026 Load the saved active long-format confusion-matrix summary table from disk.

summary_long_df = summary_long_df.copy()  # 03.26.2026 Work from a dedicated copy of the active long-format confusion-matrix summary table.
summary_long_df["model"] = summary_long_df["model"].astype(str)  # 03.26.2026 Normalize the saved model names as text.
summary_long_df["group"] = summary_long_df["group"].astype(str)  # 03.26.2026 Normalize the saved group names as text.
summary_long_df["true_label"] = summary_long_df["true_label"].astype(str)  # 03.26.2026 Normalize the saved true labels as text.
summary_long_df["pred_label"] = summary_long_df["pred_label"].astype(str)  # 03.26.2026 Normalize the saved predicted labels as text.
summary_long_df["percent_of_true_class"] = pd.to_numeric(summary_long_df["percent_of_true_class"], errors="coerce")  # 03.26.2026 Normalize the saved row-normalized percentages as numeric values.
summary_long_df["final_test_accuracy"] = pd.to_numeric(summary_long_df["final_test_accuracy"], errors="coerce")  # 03.26.2026 Normalize the saved final-test accuracies as numeric values.

selected_model_group_pairs = set([("CNN", str(group_name)) for group_name in SELECTED_GROUPS])  # 03.26.2026 Build the selected CNN model-group pairs that must exist in the active long summary.
available_model_group_pairs = set(summary_long_df[["model", "group"]].drop_duplicates().itertuples(index=False, name=None))  # 03.26.2026 Build the model-group pairs actually available in the active long summary.
missing_pairs = sorted(selected_model_group_pairs - available_model_group_pairs)  # 03.26.2026 Find any selected CNN groups missing from the active long summary.
if len(missing_pairs) > 0:  # 03.26.2026 Stop if any selected CNN group is missing from the active long summary.
    raise RuntimeError("Active long-format confusion-matrix rows are missing for: " + ", ".join([f"{model_name} / {group_name}" for model_name, group_name in missing_pairs]))  # 03.26.2026 Explain which selected pairs are missing.

accuracy_map = {}  # 03.26.2026 Store one saved final-test accuracy percentage per selected CNN group.
for _, accuracy_row in summary_long_df[["model", "group", "final_test_accuracy"]].drop_duplicates().iterrows():  # 03.26.2026 Read each unique saved model-group accuracy once.
    accuracy_map[(str(accuracy_row["model"]), str(accuracy_row["group"]))] = float(accuracy_row["final_test_accuracy"]) * 100.0  # 03.26.2026 Convert the saved final-test accuracy into percentage units.

def sanitize_name(name_text):  # 03.26.2026 Convert free text into a filename-safe string.
    return "".join([character if character.isalnum() or character in ["-", "_"] else "_" for character in str(name_text)])  # 03.26.2026 Keep only safe filename characters and replace everything else with underscores.

def build_percent_matrix_from_long(long_df, group_name):  # 03.26.2026 Rebuild one row-normalized percentage matrix from the saved long-format confusion summary for one CNN group.
    sub_df = long_df.loc[(long_df["model"].astype(str) == "CNN") & (long_df["group"].astype(str) == str(group_name)), ["true_label", "pred_label", "percent_of_true_class"]].copy()  # 03.26.2026 Keep only the rows for this one CNN group.
    if len(sub_df) == 0:  # 03.26.2026 Stop if the requested CNN group rows are missing.
        raise RuntimeError("No long-format confusion-matrix rows were found for CNN / " + str(group_name))  # 03.26.2026 Explain which selected group is missing.
    matrix_df = sub_df.pivot(index="true_label", columns="pred_label", values="percent_of_true_class").reindex(index=EVAL_CLASS_ORDER, columns=EVAL_CLASS_ORDER).fillna(0.0)  # 03.26.2026 Rebuild the row-normalized percentage matrix in the active evaluation order.
    return matrix_df.to_numpy(dtype=np.float64)  # 03.26.2026 Return the rebuilt percentage matrix as a NumPy array.

def save_difference_matrix_png(diff_matrix, out_path, title_text, annotate_numbers, color_limit):  # 03.26.2026 Save one difference matrix as a PNG using the requested reversed PuOr colormap.
    fig, ax = plt.subplots(figsize=(7, 7))  # 03.26.2026 Create one square figure for the difference matrix.
    norm = mcolors.TwoSlopeNorm(vmin=-float(color_limit), vcenter=0.0, vmax=float(color_limit))  # 03.26.2026 Center the color scale at zero and use the shared symmetric min and max.
    image = ax.imshow(diff_matrix, cmap="PuOr_r", norm=norm)  # 03.26.2026 Draw the difference matrix using the requested reversed PuOr colormap.
    ax.set_xticks(np.arange(len(EVAL_CLASS_ORDER)))  # 03.26.2026 Put one tick at each predicted-label column in the active evaluation order.
    ax.set_yticks(np.arange(len(EVAL_CLASS_ORDER)))  # 03.26.2026 Put one tick at each true-label row in the active evaluation order.
    ax.set_xticklabels(EVAL_CLASS_ORDER, rotation=45, ha="right", rotation_mode="anchor")  # 03.26.2026 Show the predicted labels on the x axis.
    ax.set_yticklabels(EVAL_CLASS_ORDER)  # 03.26.2026 Show the true labels on the y axis.
    ax.set_xlabel("Predicted label")  # 03.26.2026 Label the x axis.
    ax.set_ylabel("True label")  # 03.26.2026 Label the y axis.
    ax.set_title(title_text)  # 03.26.2026 Add the descriptive figure title.
    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)  # 03.26.2026 Add the color scale bar to the figure.
    colorbar.set_label("Difference in percent of true class (percentage points)")  # 03.26.2026 Label the colorbar with the meaning of the values.
    if annotate_numbers:  # 03.26.2026 Write the numeric difference into each cell when requested.
        for row_idx in range(diff_matrix.shape[0]):  # 03.26.2026 Loop over the true-label rows.
            for col_idx in range(diff_matrix.shape[1]):  # 03.26.2026 Loop over the predicted-label columns.
                value = float(diff_matrix[row_idx, col_idx])  # 03.26.2026 Read the current difference value.
                text_color = "white" if abs(value) >= float(color_limit) * 0.45 else "black"  # 03.26.2026 Choose a readable text color for the current background shade.
                ax.text(col_idx, row_idx, f"{value:.1f}", ha="center", va="center", color=text_color, fontsize=10)  # 03.26.2026 Write the rounded difference value into the cell.
    fig.subplots_adjust(left=0.22, bottom=0.20, right=0.92, top=0.90)  # 03.26.2026 Reserve room so labels and title are not clipped.
    fig.savefig(out_path, dpi=150)  # 03.26.2026 Save the difference-matrix figure to disk.
    plt.close(fig)  # 03.26.2026 Close the figure so notebook memory does not keep growing.

def display_difference_matrix_inline(diff_matrix, title_text, color_limit):  # 03.26.2026 Display one annotated difference matrix inline using the same requested reversed PuOr colormap.
    fig, ax = plt.subplots(figsize=(7, 7))  # 03.26.2026 Create one square figure for inline display.
    norm = mcolors.TwoSlopeNorm(vmin=-float(color_limit), vcenter=0.0, vmax=float(color_limit))  # 03.26.2026 Center the color scale at zero and use the shared symmetric min and max.
    image = ax.imshow(diff_matrix, cmap="PuOr_r", norm=norm)  # 03.26.2026 Draw the difference matrix using the requested reversed PuOr colormap.
    ax.set_xticks(np.arange(len(EVAL_CLASS_ORDER)))  # 03.26.2026 Put one tick at each predicted-label column in the active evaluation order.
    ax.set_yticks(np.arange(len(EVAL_CLASS_ORDER)))  # 03.26.2026 Put one tick at each true-label row in the active evaluation order.
    ax.set_xticklabels(EVAL_CLASS_ORDER, rotation=45, ha="right", rotation_mode="anchor")  # 03.26.2026 Show the predicted labels on the x axis.
    ax.set_yticklabels(EVAL_CLASS_ORDER)  # 03.26.2026 Show the true labels on the y axis.
    ax.set_xlabel("Predicted label")  # 03.26.2026 Label the x axis.
    ax.set_ylabel("True label")  # 03.26.2026 Label the y axis.
    ax.set_title(title_text)  # 03.26.2026 Add the descriptive figure title.
    colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)  # 03.26.2026 Add the color scale bar to the figure.
    colorbar.set_label("Difference in percent of true class (percentage points)")  # 03.26.2026 Label the colorbar with the meaning of the values.
    for row_idx in range(diff_matrix.shape[0]):  # 03.26.2026 Loop over the true-label rows.
        for col_idx in range(diff_matrix.shape[1]):  # 03.26.2026 Loop over the predicted-label columns.
            value = float(diff_matrix[row_idx, col_idx])  # 03.26.2026 Read the current difference value.
            text_color = "white" if abs(value) >= float(color_limit) * 0.45 else "black"  # 03.26.2026 Choose a readable text color for the current background shade.
            ax.text(col_idx, row_idx, f"{value:.1f}", ha="center", va="center", color=text_color, fontsize=10)  # 03.26.2026 Write the rounded difference value into the cell.
    fig.subplots_adjust(left=0.22, bottom=0.20, right=0.92, top=0.90)  # 03.26.2026 Reserve room so labels and title are not clipped.
    plt.show()  # 03.26.2026 Display the annotated difference-matrix figure inline.
    plt.close(fig)  # 03.26.2026 Close the figure so notebook memory does not keep growing.

difference_jobs = []  # 03.26.2026 Collect every difference-matrix job first so one shared symmetric color scale can be computed across the whole set.
difference_long_rows = []  # 03.26.2026 Collect one long-format numeric row per difference-matrix cell.
for group_name_a, group_name_b in itertools.combinations([str(group_name) for group_name in SELECTED_GROUPS], 2):  # 03.26.2026 Build each within-model pair once across the selected CNN groups.
    accuracy_a = float(accuracy_map[("CNN", str(group_name_a))])  # 03.26.2026 Read the saved final-test accuracy percentage for the first selected group.
    accuracy_b = float(accuracy_map[("CNN", str(group_name_b))])  # 03.26.2026 Read the saved final-test accuracy percentage for the second selected group.
    if accuracy_a < accuracy_b:  # 03.26.2026 Put the lower-accuracy group first when the first group has lower accuracy.
        low_group_name = str(group_name_a)  # 03.26.2026 Save the lower-accuracy group name.
        high_group_name = str(group_name_b)  # 03.26.2026 Save the higher-accuracy group name.
        low_accuracy = float(accuracy_a)  # 03.26.2026 Save the lower accuracy percentage.
        high_accuracy = float(accuracy_b)  # 03.26.2026 Save the higher accuracy percentage.
    elif accuracy_b < accuracy_a:  # 03.26.2026 Put the lower-accuracy group first when the second group has lower accuracy.
        low_group_name = str(group_name_b)  # 03.26.2026 Save the lower-accuracy group name.
        high_group_name = str(group_name_a)  # 03.26.2026 Save the higher-accuracy group name.
        low_accuracy = float(accuracy_b)  # 03.26.2026 Save the lower accuracy percentage.
        high_accuracy = float(accuracy_a)  # 03.26.2026 Save the higher accuracy percentage.
    else:  # 03.26.2026 Break exact ties alphabetically so the subtraction direction stays deterministic.
        low_group_name = min(str(group_name_a), str(group_name_b))  # 03.26.2026 Use alphabetical order for the first group when accuracies tie exactly.
        high_group_name = max(str(group_name_a), str(group_name_b))  # 03.26.2026 Use alphabetical order for the second group when accuracies tie exactly.
        low_accuracy = float(accuracy_map[("CNN", str(low_group_name))])  # 03.26.2026 Read the first tie group's saved accuracy percentage.
        high_accuracy = float(accuracy_map[("CNN", str(high_group_name))])  # 03.26.2026 Read the second tie group's saved accuracy percentage.
    low_matrix = build_percent_matrix_from_long(summary_long_df, low_group_name)  # 03.26.2026 Rebuild the row-normalized percentage matrix for the lower-accuracy group.
    high_matrix = build_percent_matrix_from_long(summary_long_df, high_group_name)  # 03.26.2026 Rebuild the row-normalized percentage matrix for the higher-accuracy group.
    diff_matrix = low_matrix - high_matrix  # 03.26.2026 Subtract the higher-accuracy matrix from the lower-accuracy matrix to get the difference matrix.
    np.fill_diagonal(diff_matrix, 0.0)  # 03.26.2026 Force the diagonal to zero so the figures show error differences only.
    comparison_name = str(low_group_name) + "_minus_" + str(high_group_name)  # 03.26.2026 Build the readable comparison name in the requested lower-minus-higher direction.
    difference_jobs.append({"group_low_accuracy": str(low_group_name), "group_high_accuracy": str(high_group_name), "accuracy_low_percent": float(low_accuracy), "accuracy_high_percent": float(high_accuracy), "comparison_name": str(comparison_name), "diff_matrix": diff_matrix.copy()})  # 03.26.2026 Save this difference-matrix job for later color-scale calculation and figure generation.
    for row_idx, true_label in enumerate(EVAL_CLASS_ORDER):  # 03.26.2026 Save one numeric row per true-predicted difference-matrix cell.
        for col_idx, pred_label in enumerate(EVAL_CLASS_ORDER):  # 03.26.2026 Save one numeric row per true-predicted difference-matrix cell.
            difference_long_rows.append({"source_type": LABEL_SOURCE_SUFFIX, "model": "CNN", "group_low_accuracy": str(low_group_name), "group_high_accuracy": str(high_group_name), "accuracy_low_percent": float(low_accuracy), "accuracy_high_percent": float(high_accuracy), "comparison_name": str(comparison_name), "true_label": str(true_label), "pred_label": str(pred_label), "difference_percent_of_true_class": float(diff_matrix[row_idx, col_idx])})  # 03.26.2026 Save this one numeric difference value for the long-format CSV.

if len(difference_jobs) == 0:  # 03.26.2026 Stop if no within-model group pairs were available to compare.
    raise RuntimeError("No within-model group pairs were available, so no difference matrices could be built.")  # 03.26.2026 Explain why this cell produced no outputs.

all_difference_values = []  # 03.26.2026 Collect all off-diagonal difference values so one shared symmetric color scale can be fitted across the whole set.
for job in difference_jobs:  # 03.26.2026 Read each saved difference-matrix job once.
    current_matrix = job["diff_matrix"].copy()  # 03.26.2026 Work on a copy of the current difference matrix.
    current_mask = ~np.eye(current_matrix.shape[0], dtype=bool)  # 03.26.2026 Build the off-diagonal mask so only error-difference cells contribute to the shared color scale.
    all_difference_values.extend(current_matrix[current_mask].ravel().tolist())  # 03.26.2026 Add all off-diagonal values from this matrix to the global list.

global_absmax = float(np.max(np.abs(np.asarray(all_difference_values, dtype=np.float64)))) if len(all_difference_values) > 0 else 0.0  # 03.26.2026 Find the largest absolute off-diagonal difference value across all matrices.
global_color_limit = max(0.1, float(np.ceil(global_absmax * 10.0) / 10.0))  # 03.26.2026 Round the shared absolute color limit up to one decimal place and keep a small positive fallback.

difference_long_df = pd.DataFrame(difference_long_rows)  # 03.26.2026 Convert the collected long-format numeric difference rows into one dataframe.
difference_long_df.to_csv(difference_long_csv_path, index=False)  # 03.26.2026 Save the long-format numeric difference CSV to disk.

comparison_summary_rows = []  # 03.26.2026 Collect one short summary row per saved difference matrix.
for job in difference_jobs:  # 03.26.2026 Save and display each difference matrix using the shared symmetric color scale.
    low_group_name = str(job["group_low_accuracy"])  # 03.26.2026 Read the lower-accuracy group name for this job.
    high_group_name = str(job["group_high_accuracy"])  # 03.26.2026 Read the higher-accuracy group name for this job.
    low_accuracy = float(job["accuracy_low_percent"])  # 03.26.2026 Read the lower accuracy percentage for this job.
    high_accuracy = float(job["accuracy_high_percent"])  # 03.26.2026 Read the higher accuracy percentage for this job.
    diff_matrix = job["diff_matrix"].copy()  # 03.26.2026 Read the prepared difference matrix for this job.
    safe_low_group_name = sanitize_name(low_group_name)  # 03.26.2026 Convert the lower-accuracy group name into a filename-safe string.
    safe_high_group_name = sanitize_name(high_group_name)  # 03.26.2026 Convert the higher-accuracy group name into a filename-safe string.
    annotated_png_path = difference_output_dir / f"{LABEL_SOURCE_SUFFIX}_difference_matrix_CNN_{safe_low_group_name}_minus_{safe_high_group_name}_annotated.png"  # 03.26.2026 Build the annotated difference-matrix PNG path for this job.
    blank_png_path = difference_output_dir / f"{LABEL_SOURCE_SUFFIX}_difference_matrix_CNN_{safe_low_group_name}_minus_{safe_high_group_name}_blank.png"  # 03.26.2026 Build the blank difference-matrix PNG path for this job.
    title_text = f"{LABEL_SOURCE_SUFFIX} | {EVAL_MODE_SUFFIX} | CNN | {low_group_name} minus {high_group_name}\nrow-normalized error difference (percentage points), diagonal set to 0"  # 03.26.2026 Build the descriptive title for this difference-matrix job.
    save_difference_matrix_png(diff_matrix, annotated_png_path, title_text, True, global_color_limit)  # 03.26.2026 Save the annotated difference-matrix figure to disk.
    save_difference_matrix_png(diff_matrix, blank_png_path, title_text, False, global_color_limit)  # 03.26.2026 Save the blank difference-matrix figure to disk.
    comparison_summary_rows.append({"source_type": LABEL_SOURCE_SUFFIX, "model": "CNN", "group_low_accuracy": str(low_group_name), "group_high_accuracy": str(high_group_name), "accuracy_low_percent": float(low_accuracy), "accuracy_high_percent": float(high_accuracy), "annotated_png_path": str(annotated_png_path), "blank_png_path": str(blank_png_path)})  # 03.26.2026 Save one short summary row for this difference matrix.
    print()  # 03.26.2026 Add a blank line before each inline figure.
    print(f"{LABEL_SOURCE_SUFFIX} difference matrix — {low_group_name} minus {high_group_name}")  # 03.26.2026 Label the current inline difference-matrix figure section.
    print(f"Lower accuracy: {low_accuracy:.2f}%   Higher accuracy: {high_accuracy:.2f}%")  # 03.26.2026 Show the accuracy values that determined the subtraction order.
    display_difference_matrix_inline(diff_matrix, title_text, global_color_limit)  # 03.26.2026 Display the annotated difference-matrix figure inline.

difference_summary_df = pd.DataFrame(comparison_summary_rows).sort_values(["source_type", "model", "group_low_accuracy", "group_high_accuracy"], kind="mergesort").reset_index(drop=True)  # 03.26.2026 Build the one-row-per-matrix difference-summary table.
difference_summary_df.to_csv(difference_summary_csv_path, index=False)  # 03.26.2026 Save the short difference-summary CSV to disk.

ACTIVE_DIFFERENCE_SUMMARY_DF = difference_summary_df.copy()  # 03.26.2026 Keep the active difference-summary table available for later cells.
ACTIVE_DIFFERENCE_SUMMARY_PATH = difference_summary_csv_path  # 03.26.2026 Keep the saved active difference-summary path available for later cells.
ACTIVE_DIFFERENCE_LONG_PATH = difference_long_csv_path  # 03.26.2026 Keep the saved active long numeric difference path available for later cells.
ACTIVE_DIFFERENCE_OUTPUT_DIR = difference_output_dir  # 03.26.2026 Keep the active difference-output folder available for later cells.

print()  # 03.26.2026 Add a blank line before the final summary output.
print("Active difference matrices saved in:", difference_output_dir.resolve())  # 03.26.2026 Show where the active difference-matrix outputs were saved.
print("Global symmetric color limit used for all difference matrices:", f"±{global_color_limit:.1f}")  # 03.26.2026 Show the shared symmetric color scale used across the whole set.
print("Long numeric difference CSV saved to:", difference_long_csv_path)  # 03.26.2026 Show where the long numeric difference CSV was saved.
print("Difference-matrix summary CSV saved to:", difference_summary_csv_path)  # 03.26.2026 Show where the short difference-summary CSV was saved.
display(difference_summary_df)  # 03.26.2026 Display the short difference-summary table inline.


In [ ]:
# Cell 14 — Final output summary and saved-file manifest  # 03.26.2026 Generalize this cell so it summarizes the active evaluation mode without requiring human-review files in automated-label or reload-only runs.
# Collect the key saved outputs for the active evaluation mode, build one manifest table, save it, and display the most important final summaries in one place.  # 03.26.2026 Explain the purpose of this generalized final-summary cell.

if "USER_CONFIG" not in globals() or len(USER_CONFIG) == 0:  # 03.26.2026 Require saved user choices before this cell can run.
    raise RuntimeError("Please run Cell 3 and click 'Save choices' before running this cell.")  # 03.26.2026 Explain how to provide the required settings first.

SELECTED_GROUPS = list(USER_CONFIG["groups"])  # 03.26.2026 Keep the selected CNN groups available as a direct global variable in this cell.
NOTEBOOK_OUTPUT_DIR = Path(USER_CONFIG["output_dir"])  # 03.26.2026 Keep the selected notebook output directory available as a direct global variable in this cell.
EVAL_MODE_SUFFIX = str(USER_CONFIG["eval_mode_suffix"])  # 03.26.2026 Read the short gray-handling mode label from the saved user choices.
LABEL_SOURCE_SUFFIX = str(USER_CONFIG["label_source_suffix"])  # 03.26.2026 Read the short label-source mode label from the saved user choices.
NOTEBOOK_SUMMARY_FILE_NAME = str(USER_CONFIG["notebook_summary_file_name"])  # 03.26.2026 Read the active notebook-level long-summary filename from the saved user choices.
EVAL_OUTPUT_FOLDER_NAME = str(USER_CONFIG["eval_output_folder_name"])  # 03.26.2026 Read the active figure-output folder name from the saved user choices.
USE_HUMAN_REVIEW = bool(USER_CONFIG["use_human_review"])  # 03.26.2026 Read whether the active evaluation mode uses human-reviewed labels.
LOAD_SAVED_RESULTS_ONLY = bool(USER_CONFIG["load_saved_results_only"])  # 03.26.2026 Read whether this notebook run reloaded saved outputs only.

summaries_dir = NOTEBOOK_OUTPUT_DIR / "summaries"  # 03.26.2026 Build the notebook-level summaries folder path.
eval_output_dir = NOTEBOOK_OUTPUT_DIR / EVAL_OUTPUT_FOLDER_NAME  # 03.26.2026 Build the active evaluation-output folder path.
difference_output_dir = eval_output_dir / "difference_matrices"  # 03.26.2026 Build the active difference-matrix output folder path.
summary_long_path = summaries_dir / NOTEBOOK_SUMMARY_FILE_NAME  # 03.26.2026 Build the active notebook-level long-summary path.
accuracy_summary_path = eval_output_dir / f"{LABEL_SOURCE_SUFFIX}_{EVAL_MODE_SUFFIX}_accuracy_summary.csv"  # 03.26.2026 Build the active accuracy-summary path.
difference_long_path = difference_output_dir / "difference_matrices_long.csv"  # 03.26.2026 Build the active long numeric difference path.
difference_summary_path = difference_output_dir / "difference_matrices_summary.csv"  # 03.26.2026 Build the active short difference-summary path.

manifest_rows = []  # 03.26.2026 Collect one row per saved output file so the final manifest can be displayed and saved.

def add_manifest_row(file_path, category_name, description_text):  # 03.26.2026 Define one small helper that adds one file row to the manifest.
    file_path = Path(file_path)  # 03.26.2026 Convert the incoming file path into a Path object.
    manifest_rows.append({"category": str(category_name), "description": str(description_text), "path": str(file_path), "exists": bool(file_path.exists()), "size_bytes": int(file_path.stat().st_size) if file_path.exists() else np.nan})  # 03.26.2026 Save one readable manifest row for this file.

add_manifest_row(summary_long_path, "summaries", "Active long-format confusion-matrix summary table")  # 03.26.2026 Add the active long-format confusion summary to the manifest.
add_manifest_row(accuracy_summary_path, "evaluation_outputs", "Active accuracy-summary table")  # 03.26.2026 Add the active accuracy-summary table to the manifest.
add_manifest_row(difference_long_path, "difference_matrices", "Active long-format numeric difference table")  # 03.26.2026 Add the active long numeric difference table to the manifest.
add_manifest_row(difference_summary_path, "difference_matrices", "Active short difference-summary table")  # 03.26.2026 Add the active short difference-summary table to the manifest.

for group_name in SELECTED_GROUPS:  # 03.26.2026 Add the per-group active confusion-matrix PNG files for each selected CNN group.
    safe_group_name = "".join([character if character.isalnum() or character in ["-", "_"] else "_" for character in str(group_name)])  # 03.26.2026 Convert the selected group name into a filename-safe string.
    add_manifest_row(eval_output_dir / f"{LABEL_SOURCE_SUFFIX}_cm_pct_CNN_{safe_group_name}_{EVAL_MODE_SUFFIX}_annotated.png", "evaluation_outputs", f"Annotated confusion matrix for CNN / {group_name}")  # 03.26.2026 Add the annotated active confusion-matrix PNG for this selected group.
    add_manifest_row(eval_output_dir / f"{LABEL_SOURCE_SUFFIX}_cm_pct_CNN_{safe_group_name}_{EVAL_MODE_SUFFIX}_blank.png", "evaluation_outputs", f"Blank confusion matrix for CNN / {group_name}")  # 03.26.2026 Add the blank active confusion-matrix PNG for this selected group.

for group_name_a, group_name_b in itertools.combinations([str(group_name) for group_name in SELECTED_GROUPS], 2):  # 03.26.2026 Add the per-pair active difference-matrix PNG files for each selected group pair.
    safe_low_group_name = "".join([character if character.isalnum() or character in ["-", "_"] else "_" for character in str(group_name_a)])  # 03.26.2026 Build a filename-safe string from the first selected group name.
    safe_high_group_name = "".join([character if character.isalnum() or character in ["-", "_"] else "_" for character in str(group_name_b)])  # 03.26.2026 Build a filename-safe string from the second selected group name.
    add_manifest_row(difference_output_dir / f"{LABEL_SOURCE_SUFFIX}_difference_matrix_CNN_{safe_low_group_name}_minus_{safe_high_group_name}_annotated.png", "difference_matrices", f"Annotated difference matrix for CNN / {group_name_a} minus {group_name_b}")  # 03.26.2026 Add one possible annotated difference-matrix PNG path for this selected pair.
    add_manifest_row(difference_output_dir / f"{LABEL_SOURCE_SUFFIX}_difference_matrix_CNN_{safe_low_group_name}_minus_{safe_high_group_name}_blank.png", "difference_matrices", f"Blank difference matrix for CNN / {group_name_a} minus {group_name_b}")  # 03.26.2026 Add one possible blank difference-matrix PNG path for this selected pair.
    add_manifest_row(difference_output_dir / f"{LABEL_SOURCE_SUFFIX}_difference_matrix_CNN_{safe_high_group_name}_minus_{safe_low_group_name}_annotated.png", "difference_matrices", f"Annotated difference matrix for CNN / {group_name_b} minus {group_name_a}")  # 03.26.2026 Add the opposite subtraction-order annotated PNG path because the saved order depends on accuracy.
    add_manifest_row(difference_output_dir / f"{LABEL_SOURCE_SUFFIX}_difference_matrix_CNN_{safe_high_group_name}_minus_{safe_low_group_name}_blank.png", "difference_matrices", f"Blank difference matrix for CNN / {group_name_b} minus {group_name_a}")  # 03.26.2026 Add the opposite subtraction-order blank PNG path because the saved order depends on accuracy.

review_scope_name = ""  # 03.26.2026 Start with a blank active review-scope name and fill it only when human-review outputs are available.
if USE_HUMAN_REVIEW:  # 03.26.2026 Look for the active review-scope name only when the evaluation mode uses human-reviewed labels.
    if "REVIEW_SCOPE_NAME" in globals() and str(REVIEW_SCOPE_NAME) != "":  # 03.26.2026 Prefer the in-memory review-scope name when the human-review cells were run in this kernel.
        review_scope_name = str(REVIEW_SCOPE_NAME)  # 03.26.2026 Reuse the in-memory review-scope name directly.
    elif summary_long_path.exists():  # 03.26.2026 Otherwise try to recover the review-scope name from the saved active confusion summary.
        summary_probe_df = pd.read_csv(summary_long_path)  # 03.26.2026 Load the saved active confusion summary to recover the review-scope name.
        if "review_scope_name" in summary_probe_df.columns:  # 03.26.2026 Use the saved review-scope name column when it exists.
            non_blank_scope_names = [str(value).strip() for value in summary_probe_df["review_scope_name"].dropna().tolist() if str(value).strip() != ""]  # 03.26.2026 Keep only non-blank saved review-scope names.
            if len(non_blank_scope_names) > 0:  # 03.26.2026 Reuse the first non-blank saved review-scope name when one exists.
                review_scope_name = non_blank_scope_names[0]  # 03.26.2026 Recover the active review-scope name from the saved summary.
    if review_scope_name != "":  # 03.26.2026 Add the key human-review files only when a review-scope name is available.
        scope_dir = NOTEBOOK_OUTPUT_DIR / "review_scopes" / review_scope_name  # 03.26.2026 Build the active review-scope folder path from the recovered review-scope name.
        add_manifest_row(scope_dir / "scope_human_labels.csv", "human_review_scope", "Combined current-scope human labels")  # 03.26.2026 Add the combined scope human-label file to the manifest.
        add_manifest_row(scope_dir / "scope_human_labels_new_only.csv", "human_review_scope", "New-only current-scope human labels")  # 03.26.2026 Add the new-only scope human-label file to the manifest.
        add_manifest_row(scope_dir / "scope_predictions_long.csv", "human_review_scope", "Long prediction table for the shared review scope")  # 03.26.2026 Add the shared-scope long prediction table to the manifest.
        add_manifest_row(scope_dir / "tables" / "final_shared_human_labels.csv", "human_review_scope", "Final shared human-label table used for hybrid evaluation")  # 03.26.2026 Add the finalized shared human-label table to the manifest.
        add_manifest_row(scope_dir / "tables" / "shared_scope_with_final_human_labels.csv", "human_review_scope", "Shared scope table with original and final human labels")  # 03.26.2026 Add the merged shared-scope original-versus-human label table to the manifest.
        add_manifest_row(scope_dir / "tables" / "hybrid_scope_summary.csv", "human_review_scope", "One-row summary of the finalized shared review scope")  # 03.26.2026 Add the finalized shared-scope summary table to the manifest.

MANIFEST_DF = pd.DataFrame(manifest_rows).drop_duplicates(subset=["path", "description"], keep="first")  # 03.26.2026 Convert the collected manifest rows into one dataframe and drop exact duplicate file rows.
MANIFEST_DF = MANIFEST_DF.sort_values(["category", "description", "path"], kind="mergesort").reset_index(drop=True)  # 03.26.2026 Put the manifest rows into a stable readable order.
manifest_path = NOTEBOOK_OUTPUT_DIR / "output_manifest.csv"  # 03.26.2026 Build the saved path for the final output-manifest CSV.
MANIFEST_DF.to_csv(manifest_path, index=False)  # 03.26.2026 Save the final output-manifest CSV to disk.

missing_manifest_df = MANIFEST_DF.loc[~MANIFEST_DF["exists"]].copy()  # 03.26.2026 Keep only the manifest rows whose files do not exist yet.
existing_manifest_df = MANIFEST_DF.loc[MANIFEST_DF["exists"]].copy()  # 03.26.2026 Keep only the manifest rows whose files do exist.

n_test_total = np.nan  # 03.26.2026 Start the active total test-image count as missing and fill it only when the active long summary exists.
n_human_reviewed_images = 0  # 03.26.2026 Start the active human-reviewed-image count at zero and fill it only when the active long summary provides it.
if summary_long_path.exists():  # 03.26.2026 Read the active long summary when it exists so the final notebook summary can report key counts.
    final_probe_df = pd.read_csv(summary_long_path)  # 03.26.2026 Load the active long-format confusion summary for the notebook-level final summary.
    if "n_test_images_total" in final_probe_df.columns and len(final_probe_df) > 0:  # 03.26.2026 Recover the total test-image count from the active long summary when available.
        n_test_total = pd.to_numeric(final_probe_df["n_test_images_total"], errors="coerce").dropna().iloc[0] if len(pd.to_numeric(final_probe_df["n_test_images_total"], errors="coerce").dropna()) > 0 else np.nan  # 03.26.2026 Read the first valid total test-image count from the active long summary.
    if "n_human_reviewed_images" in final_probe_df.columns and len(final_probe_df) > 0:  # 03.26.2026 Recover the human-reviewed-image count from the active long summary when available.
        n_human_reviewed_images = int(pd.to_numeric(final_probe_df["n_human_reviewed_images"], errors="coerce").fillna(0).max())  # 03.26.2026 Read the maximum saved human-reviewed-image count from the active long summary.

FINAL_NOTEBOOK_SUMMARY_DF = pd.DataFrame([{"selected_groups": " | ".join([str(group_name) for group_name in SELECTED_GROUPS]), "eval_choice": str(USER_CONFIG["eval_choice"]), "combine_gray_classes": bool(USER_CONFIG["combine_gray_classes"]), "use_human_review": USE_HUMAN_REVIEW, "use_previous_human_labels": bool(USER_CONFIG["use_previous_human_labels"]), "load_saved_results_only": LOAD_SAVED_RESULTS_ONLY, "eval_mode_suffix": EVAL_MODE_SUFFIX, "label_source_suffix": LABEL_SOURCE_SUFFIX, "review_scope_name": review_scope_name, "n_test_images_total": n_test_total, "n_human_reviewed_images": int(n_human_reviewed_images), "n_manifest_files_total": len(MANIFEST_DF), "n_manifest_files_found": int(MANIFEST_DF["exists"].sum()), "n_manifest_files_missing": int((~MANIFEST_DF["exists"]).sum())}])  # 03.26.2026 Build the one-row notebook-level final summary table for the active evaluation mode.
final_summary_path = NOTEBOOK_OUTPUT_DIR / "final_notebook_summary.csv"  # 03.26.2026 Build the saved path for the notebook-level final summary CSV.
FINAL_NOTEBOOK_SUMMARY_DF.to_csv(final_summary_path, index=False)  # 03.26.2026 Save the notebook-level final summary CSV to disk.

print("Notebook outputs summarized successfully.")  # 03.26.2026 Confirm that the final manifest and notebook summary were created.
print("Output manifest saved to:", manifest_path)  # 03.26.2026 Show where the output-manifest CSV was saved.
print("Final notebook summary saved to:", final_summary_path)  # 03.26.2026 Show where the notebook-level final summary CSV was saved.
print("Notebook output directory:", NOTEBOOK_OUTPUT_DIR.resolve())  # 03.26.2026 Show the active notebook output directory.
display(FINAL_NOTEBOOK_SUMMARY_DF)  # 03.26.2026 Display the one-row notebook-level final summary table.
display(existing_manifest_df)  # 03.26.2026 Display the manifest rows for files that currently exist.
if len(missing_manifest_df) > 0:  # 03.26.2026 Display the missing manifest rows only when any manifest files are still missing.
    print("Files listed below were not found yet.")  # 03.26.2026 Explain what the next displayed table means.
    display(missing_manifest_df)  # 03.26.2026 Display the manifest rows for files that are still missing.
